# NB-FLUX bounded A100 spin-resolved tuning chain

Select **Runtime → Change runtime type → A100 GPU**, then choose **Run all**.

This notebook runs a real, finite Markov chain: 200 thermal cycles followed by 48 saved
configurations on 16³×20 at β=6.0625. Every saved configuration measures the fixed-radius
ρ={0.30,0.45,0.60} spin-resolved source registry; topology is measured every fourth
configuration. Immutable raw chunks and two-slot checkpoints are written to Google Drive when
available, and an interrupted run resumes only after verifying all hashes.

This is deliberately a **one-chain tuning dataset**, not a production ensemble. It cannot support
a continuum, particle, residue, or final spectroscopy claim. At completion it downloads a ZIP
containing raw streams, QC, summary, provenance, and the active resume checkpoint.


In [ ]:
from pathlib import Path
import base64, hashlib, importlib, json, os, platform, shutil, subprocess, sys, zipfile, zlib
import numpy as np

EMBEDDED_PAYLOADS = {'NB_FLUX_SPIN_RESOLVED_engine.py': 'c-ri}Yn$6Pk|_MWe+5VL?0ZC8Qn%E$<LN|sVmr=d*Ksnolbn6@Y8kdfb<>tejY!$u?(P46>ViuF1SPeToU^y=Y+EFOLZJW@3Wd7-^)JWUYIA&^7spxgK3ZSbmt}E!)ai6yuk)h6$*OYsKAZQVMShjd`<ry0Z>uP~s<WcX%OWZlQGJ<3n{1utn`|DveA@r!#n1mc%8HA;$OcD8Z!YsHT9xzdGK=QfGC$8YX`L;vqkOe4H+2=k<7Ku1C}|Nb%FVlq0KQL``5X$)qwO01joxLOB3njzarE17FP|pZNb+X*(+>%^lxPeF>+9cs8${9fb(GHM3~ZI9)pnDuvI1Tf^Jtc?pzuWzEz|34gJshCXtk|s=vWSa&oe+IgQC%s;mJv~zPzsT8DP3CBIryt%K=cn$Y<#i`T@X;ytr6wmuE*u_n<o;sEvPYb0{?HLpL`iP%wJdFX!`Uy-bUA6IJDQGs~*ap~9?$a>aJL>LX3dP4r9m%`omo-|2szjNl&x32#PmG+P2V2ST+@>pWdXzvj!TEc%OO`5{_n@3SQkKCNe$(0E$&07jd8)ju|Mx5_V8X^fTAs9Kd}jfEBfvOmvPJhT8%HC31Ea#>znN6?e2Yr^?@w#@oKUzi^i)O_{wTR^ccSNSZ0_Ga(aCG`FnzSilcX6*A~o~<+Z1A{QTOmm>+oYB8bapqJt%mkn+BAHb%LQ9}34_CR_WJ{V1i@ZKQPpiB-{s8cxpU}X0zRc_E=rUWbVK8aRWzZ=YoandTHrWN@2>to(Hv;zCZ_wX&89YI67Wp!(B4}REH=5GL@){-(LQ(TJyIPl3Rvi^3(DfpnT}M@Z)qh_ux2p`n%(pYDSzW@6o^9&^uAif$#im?ENwV11F!_>1&3M3~C~In}Iy%ztHW%x3Q)T-3GOaFQt?0+MFs}MH4C$r*T~_)R%&QtE?Mgpa*XC25t=8C4^9&=*xIu^0*)oM8tPFD1JfFdcnNPU#vn4P9SYXR^rfR~BL;Q5DpAaYdP+zZc*wp*43D);Dtoie0){9=i>gq*5q-!h$Kejc#H_tz=Ynphy=ympwZC1>n%A4&PP}jH@+tvCSkc^5ow?{7qZmTc9ezMu5*IWQr>sm9L2&PoJgoa9JXqA5i@GyUBI}E!R3$_YV4{$kp{_@-JUp`NMdH(A4_dmXj#!*MJy-G4om>St)nO|Ji(dcCKsDJXfKN@w8UOo9WdHw9W=RZ8bn#DOR*zHxHYo;Ii@1e+(mrq{&>+A1dx0M4PvIL$~_1_Pn@J~Oz`tg_NUnf8P>znTwozuZ^G#H+WZ!drT;irGaPou%<;Gy{N`q}sVVK^8*9!PL6pMLY>s~?`c!H*9I!=d-_*^BR=zod^R`0+%11c?9g{MF0nFYxPU&>(z#{o~KCo;^>#e*Vq(FTZ~SWJ`W{^78v{p1(#AH%Aft*QtPatkTYz=q+)%{ybacMNUfwyWHdAKrJl${p!u8`)hoBs^4ptiytSaqevC#^S}Q#il@h)jdWpVx>)pelD~a7ew9I?r%4_?P2R%)zl%SQeo68%eti3R^xe3+Uahiv1DqZB)hq-4QFUL(8XEH6gf~e9`6`(or(@!{kD@x=CY33vn08WaS6Fg*Or>7@+bNc~g-Hf|M%Flb_$WEQuCuDUPOq0`IzNl(F^>9QBF{V{aGMNxOrhDvfa~cR@!&GMn&%h5jl1z+y9PGdjolWIKXh?XpV3+l0{4)qF9(|}ovY{A0Hh_IASPX4s@u$k1&g|z!_pWNPaDi(`Bq#G)*>vy<Xv`Mjo)nO8CD+b3FW35cRRgKFY26ia4TFQk&Cn#f0Hh&EFLh-Zf9FB`j0ztE6%EIjVUvC#&lPQi8e@;ZbHA4Bp#?GZo7?~A-Ow5tTxyIb_0AcXUU~XmgVeS2sn-f5VQ@&!H2xQ1bVH2M4G|9dj9vHzkl`o2Ux+clNUce`<Fp|RR>9DW=0(43u<aWLJ|yl1QJFTb$<B%_3Q6ne%sL#fR=1xm+UTjh{@8<AneqFWmhLSaT1(3V#4+!vpUpnLhZdQi;NqD4fzSlCnU3d*yJEmGQ8-YU;p^>^XO&zlH`N$3&Q)FqypGc&+|EmECbxoeOeP$%-aD>Rv4zi>fJowbooKgWTIG7zN1I3Qm@tkhbr3fp-21`-H$p0D58c0OJI#b2@+Jd^8wZZO=jrlSWkQ0pFoD5FIA86KeNVoq7mh)bded(&{I`<B%T%Zc+^~m{(3Xex$1z3+3o!Qg+jKh21-`w0?NIJ$)x7<wm|VCS%J*FKz?+d&fbAgcxLwg31Rw_Ur&KVIEFNsC`?RX%wYRIf1AzfDfg2$c(wGPI0!xiey9c)S>5f>ca7S%K5N1;8xTztJ^}$x)*{t_-y|Rkqa4|dd$ur%A5gYf!zxw<l0+2X*3Tk9C9UgCS7HkcO;keuwk_Tj<p&_7fS}QEFj7ApT$H@R>lR8o+(?uoR^=CN@-I=t;X<ZL8*h@m_j!S+-o_`m`Yrv<k+ITDz=Icv&0<BX@tky2i4KmV`b}%dl(J9;d6itOw-$f_GTXv1Ap|{8y2jOd$1pYF0Q6*)t;)@{gAc!g&emzZ?7H>p&9#dVF-Tr00>ssNFx$@4f%?*@p}H|o6IFr`fMZx!J?om^X7w+6D#ts(%r`mox&eh>9}FaWpTj8U^9CFs^w;#!2asLOvUMFjr+@HRm7>C7cDBFANw(RPjPas#1BZgF3l(Ennm8Yl<n}DOffu(Oxd#ChU`}6|1J4otjZppZN(u7MO4t-Gn1?<us%!<D7Qc(l?!wR84WA)lal$z=lZRf|cGP(o_dM99Ht)*!d=jSJMd7r_HhAz!Qj`Z1I;WpSAXdT&ejJ?)9-Oq7<L=ASCyGO+u&Q~zo#C#BVmX{?m-#HO`+2e6)@pMjMF$9_D8c55#DI=r6N21}27>|8xC`h!>_Ib+PIv^hUIm8Rv}4Xd+ywQPzQbHV9L3vN3B#yA4Alp%Oh_`Ab=Ugq=pk>SN;Bbp9I9?B37cf{s?OF9WfTM!*pUYjBTMRX35V4J2_*}=t+9Zggh%M%iCcJQ^QinuS8ytYrguVp@K6tB4X0_?_kodS%VniU26`^Wr56o(L;!T6)K|$l^61J?@Z`jl;4cZB3lbU$kID)$(!s#m57THkQt~}aeVCSdMIvO_UCMF^rAU5I^S8?9+bk(y6HAx*M_M&z&M3h@oA(h$|6-G_^pd45XPzLt);w9r4p3i<41RgbB}94XrYTPqP)5<J$8MD^7oHeNW3Z(I)WDRB<xYc}6xAFA*J$)*bV70_s&QaDqRlVcQ6Y|9#<xjx5W0WC6Vr2C?%htf9a3kWg9u9vP?PJj%5eZYcR`q$O7!LUj(BsuPsCm|8t6k&7DaZEHoHNXtheXO92UmhA%zef6pUJTW40`-tm~jiQ@Yy@F78F`y+$?J5*NsLnDvJsOsD6SeKhTS;+O)r0A4z)D?Sk9#bt)FH;ilu(L?E@i2cV>Ne%xlT5*v-m13%E`{~AeWW@4AV|h>?<SshEqvIORv;f=`5WAoLz7J^$)47=d7-sgvl+RY{`dZ*c`!g`E8hL-0$Kom`spAT|rv|%iH*fTh!9nG;xb9v>U%*l~py(TQlpDj-VK#@62T6`T#c}kPF&Ep{-ByfbU-j}Pdhih)Z;OiACQB>W_yW^fh!24LP>)LYB>G}3O8`;7Xw=v{NTx0!xy&;rOL38PC72*yuJodjqmQoaf}i#y;(<d7d0;dlckucXO@xMkgye#qoZy;LO+iK#+e<jQh{6h$Lpl&op!ReZX01{glIfs!!?is$hQUGGC6W=IdR6Qlubq(eNFl#9x2SMDEbsyh!MnKO(!}FaF^V-3Ea%ZuEk<V9H$=p}^s4cS*4|a*D_G}rs;8^4XjS6Fb`>$W=--Qf|F_W?s23W+ff>VAsS(F;aI&AFx7I&o6x(HWn!TDu1f-evn)?aa-sr*NnA$G`;xV$>plG}gEgHo@MJ<I#ZZmmhr$9CjU>B6}9UNH8rQ~y=RB2Pr9jamh47v?9K&ze(3FcjnHFXFZPj}~3qFb|29k+wB3f-qURpNklhsNj?(~n1~QHGU&%Je|pRqF!{n>DQSf|!L;6?K9j7M-7nEYXceireFt+tp9k;yv~D_W1QIcV1JEZa<GMV7;`panm{ox5qawf#HQiz1t?t#-O)egN(=a{^1}pw^MZA)riy)br+lRV^%12J}Zn~(VG7Wx92l+_{?DYn<tygQXQ48&Vf&&_9aL+umYQf4gZtM9i>gzC(uOc2=$8W&K4JE0S#6!GU|4zWp7@UJm(cov+UcSevXjjxc0LdO3mq_CWD<Un#0+kWVs`sZOU_yzb{wm=AF4NQTqbmvxjT2GEUdQ@@talU3IMjw@aHF8gpz4@W!4rhCMB!^nIEyNh$a2=dYg}Kik6e*GgqO74?+`u(P&X42M->AA<S{WUzF~!jp+x$SNsxp}n1JO!`o7Y4@3DFi|&IScyVAQV<h&c{*Lx{bycIDs*DHtzEx+Zx$UI4vpn1ugIk7EV^+T|6kb3|B7$1^_$k_Z|~3-K>+F#%3iIb6@OC4V6qwJy`y8cnU)UP0*wwbpCHpOBb1;)80^4Zqua~RjJ*wOc&Bg*Z>S-Uh)uAzQSds}I$F+gmCkCddSz2IM64nyHDC@QIYsITX@Wa6WSUgkv9+Cg=&V$ok=c>N8FhY>Df41L3pj2-rpLEqrRRX(&Sb%FUq}RZpLf*(J8;Mq`~~QNx?2Q&EJ+bZ_wGd_c=rfoYt=SgyleHo+&ut-q~bjmaGByz{T{YL7x~y)0LqfJ)4H88ipo?hJ+Hc^kv@Km0VVjyRAyoi(ly?9bOp7sOFe9`XV4_2Pwbc*uy%ogn0Z@nH;FJvOw7*NwYy|!uF_PcUuI%{;~O_hYe`yyWG)79d4fJBQr&arnVhUN1>8x>9nl0;k34~P4C`Y!3#tAZgUY1JNXL@#JxR~KHX*CE3%<%BeObBOf)GGPnN)zfZ{!fP<>fo4b++kKsR%(wRT-t?h9xTSC9AVGISmwi$8F@*%Wu_Pxv+yE0|L5fd-)Uvr+>e}u|~l0E&tX?<1=Bb-n$sbw;f}_lY(TJyJjd*VSpo}Y<Su%oV0n<<m>@(ug)^B>tzN937|E|$l7v28`V@<&5b9U;-VWAih7YDzb7AQ=>w{|*RyasotLY@EBXgW4kt=O65Vz}R%2aQmYe)4c8p>89rPr<zzNz-ukva*?xRW+evQWcVLYg|D|(L)4n(p;V+g(IYONq=*LZ^rsKJl!X)l`No#<FMgb#5O16bZ&7fqO@r!Y$=!0CHL2>v?-egc32kLH!>3pnrL-7b;z_0Olg+SFTnYx+xznfH3Rtvq8&sqB&#-IcvPMRK6ENy2z1=@rZvJhJxC^7CEAwkTq^!@U5$F;-OE`zSW{E40_4^Y7K#ye#JVM%`;u3pg*=*`$9ub@q#8R&-@6BvD}9{YUpqqers<#X`OF)83V(vrFn6$l{xf%#g*N@&k|l`3Ba=EUi!#*7autb&2PK($-TSqWck+IgR7NtXyA<OD@=CasNEQ;m1Ch4~P%x2VpZMLi@)!`Z!kV7u-?sMI3z@AVrWIi%wq9nTO4=>JnC7bb~-{Jw<^b25yL^p}x9R_EGxM-^%enBI=kY4$pi8v=^Ph|E5iu4!Q-5BVCau8mN#J+ZFjcz{XYu^Z;7$p$g&80u?nl`GO}~kEflOR+ITO*3CDKG602%>B2O+KlZ4B`~2CyT7AFPe@RkFDD(8<BHIucw;(RAAEK|Qu&=3Y_8~Bt(U^~CGVE%2mL{V)@>0100CG+%2K|HObL^Xuka51t<5pWGB^3I{VGIt~=y%y>1p+jjiOr^>oj|A|94J1y=A;}Xcb3cu1^8dui_U>Ct1U2)A>x4lTHXP@MegCAhq_p)7xm$v{<*Lpqi^N|4441Se1Q6mZ|Xf(7vC&R|8DvD;A6Had8_caUw}(G{EmiC@w>_$@xS8|<C{2tPZGP4J31ElH%O(w1Iv<=@b5A1#a$2?-!|rkkORKy>5j#A!sy$cbDlGLW7Mw+B{3WR8a=r}ry|x3SJ74Py7!S?ajJERSCt??<ujfERdoHa_wlNCbv-Zu)OYAb7v=%_Fm*wcIl+JF-lEX~b-ze=FwW$b81|yKy$IGb!KCeph5=vVFmR)y*KFRO%u(#G^K4(QVJ5#~Y?dV++<rO!PJxIF2<;wLUuLlRYv-%=rhLmR&sh#Zso|;NP>nHh@H#RzXaKKQN-TTQi=OqOuRRX)*XSES(=OpBy=T3zNlmxdqS3SVT)-0?!HIwo;LyIxqBooFlV@MY1D9pViPe1qtnL~7|JP_*o-PL<PN0DuEFL=IJbEC|{RN3-583$NcPQ7ouilKB#c(ild%=3@_3@Vkuf0(Uw$yk!$w7#FI(e)9su>}HxQzRClGE*Z6NLWjmZ#q6gDqg-ex;7-uME16HgO;wHOkU7g%yz_Q;eg3ih#$^!-yW7vP>NfcvpA(fu6ime?`}yXmZQaCsb}oe@9a>gnx~G+@L0(mOoJZe7>FG#Nov-E6dG1FJQZ_1`!;kQ5w61V-=cOyhraybZ)bt48`#*eR}*b2GYD4CfIX&HNscl(W`;a7H%A>bg&{CHygbMoJ@cxK7{iLP#k6>oHS?ex-@2davGm_TSMnNAI3w1G4f#0$hO0HGIC%%;@V8*37?WsHP&o<6u{qan@2y;_-%u?DNoQKXLIf5m1EqbFl%-?4e>stagW9W|2`Sf?@^Ru<qFr;q+IuA<qGxz80+54dK%H)&tAr7d8qo<WY4=8pmmw*Zpsh7g!LYI88DNLJtdo6LT}@OTLIDSAn>lUxEQ>+ckh%L$c7jQG&2n`#X=9%gsMIr)MY}>0NwXo{XJGkl%{47j~Y|^U3Pue?3S231PV&^2onTpNHxgoY*lq*F(J^LDBUC%AY-1V%cUag_5MBY1#Nby3WFv7WO)C+YUVon7HD`36anO5xw32Azm4KQaE}w#+={g;=EM@`S1ft9ZaPFEBb+{MNEHZ{)E7*quh8K$pTS0WS<VgNw21>-OmvHp)%&o*TNv5uKU1msR-(T)&=`*!%3;Q)i?YuNi9vBo#bHaup{@v`t@w?yl>ONkze&n6Kl_fv6}AyDmL25$<Qf6UNoMRz+Br`z()P(ajvF-7uA(~KTx7L&q^{6ey)J8*kjC-^#q)t(KIocFbr<QZhT+HKQwoc1wuQ3U)f(h5GG*W}0D0vG9w^FM*z%s^WW%G(Od9a_&#(K=i_b}rQm*Qm@&SmnAFe&$WE<p!d_O-!JwL(L&Cu;1EUY=mUUY4$Jl(T_B+KrF#f#jmuC^r~US!N~4`#n^o5?#?CoL8={(sTs*T~;8!E1MMR=L`xCl=k%gtjMg&LTEBG0YGby6VWS0k}-hGn{T7Otc6;7)FEHsuG&jx+{~XZG$LSSCa(k(Tg4&MiPyIvgP9C(@;&&dp&+0-%?zpsn1AfGgujfC!cf)OQozt5zwIYvX*-#?(xg)YB+iv2Q@gw6`b&aUd2a)a*O6>euZ*xaRp+o?LJoaUwm|4#6f69c0uRLesL9h_2J!h;N8dg0I*CYJhZFnKIv9VRAQ40VlM`175Xt%+w<9!uG4r9xFsX;fVBYMY0H=#kHy45H8Byf#ia`7=F}VrJh6ko(Iy$|UDB?jUW7Vabqf+uchTB$z%PI~uh+>{f(z$1DMmMHas{+ffLc$%XsR35<?=dxSH3??>tJP2R5NX|HQv<}H5&w%3s!e4%>qUM!Ke=a1BL^Ylpu-rUi4l}9qEMIz0)c42ca2qbm#iz?^B>lB|2~9(*_erEOuhrHW!yxLRy7qqG;>#a#qXcN*&w7_QM+DJfImSuN)<n{h)4xJk`T99uvX<HsfUp1A}{@9we)fN6~^n^>cKK!~1dAr~f6()*eWE#-#mUS*W0eEJ3b6lnAG&(6}RWPNYrb^}yBXyR3_w1EK|@3kW+bo%4y@9R`{>dO_5i#=PeOLM4l`(k3agE#QSd4Vw(;kYM<b1Lpl}s6_U+<(kYGvc9I>vwlP{U`<fZ|9yk5T5FWmhRk$@Z1ndee~Tgy1{>NI^9&9$C~8z7PAMy6K{RC)%Ayj5&B?Fq7QKPmBp9iK9^S|;a}1J{l1)BHU2ld_H-$koQ8A+Yi~g5yW4^$9WBoU}eu>X9@{sCsAG-%9XRSn2A|2rQPKka8OGjq4J7<pu6AFtt|4`zCEA3AmNAcT3nTQ}WEOffO!y0^xXDiW-0lq^Uqb469HDrPXdaTm2_awFfG6>kz<6(e7Ad$G&lYAPGCSbji*$CcWmmou|d;D^YM+sUQ7KJ9q_{XFRJmqanZinmWo(70k^l!Vs`tntJ;ho1@4&#>1eS!+P9=Z&^9t9Y_q98-{25YF-dDnKSkKKC<b0D6&0vsx9JRj~+p#J^C>gyAp1Mo2YLOrzFg+04|?^D2!J^{QqDsr$h^eF`?DFKeOS69U=FK`f$;=C&^w8dLfk=B-Fxz@qbBDHk@*8-g2aJJm9$KctLsB4Y`ree9yIB3mT`>DpAocicudf1%1EKz6|4_qbKuY_J#{zc=5M6hfJ6p7-BKsyUH?^F~8LbvoE<jKMcK!Ii6sBA{-lOgCEMO!POS6u}Gdl5do?6|tq|NR;@Z}9y4C>sT)*}0*Wn|@WiFmP=Wt~(NSLBP9XeH;~#;}C_WxYnRPl<c04$UqQl^s#wlS)YvpH4*f&wv5Ch3lT8hF}MencIT~Tr$M}Zo&{BZx6U`-<Au<&bnb+wuK%YaYj}`G&+vPf%7(kU)tMpLqaaqgq4nh3*|awlpw)q`nVQWeedW7p-d>b64x*d&EJ05^kZ%5j?dMq^r0XJDX6d{10s}@Cc}<SAbJ}T0RF2l;ljzB?-;ZY7^Q?bff?S?%uA^e>NbV?q8^a1yC`Z*M%omj2fopw8H*<66JkJ-(>A6zEy%m5iH(7>z;6Jum4QGi3-uJ2#P4rW?$;&xL|G0)J_6|2j67Cr)HugR(^6JuRsP##0dHuOMgHgQ2TJNaH6E@ie&<IeTQP9y!T!}~X4Tyzz5aaK>>&_Ay2WKPGG)&-BwhCwf_-+$m(H`Ln*A4=nSbfN@(AbVTKXue*G&!020f2CD#7^OwWkJ9A+=33@SfceRq{-<~=5p({Y$AcWOwv{c=QFn;2eiolZl1S?<@xr1dS!^CjUh(kh@)<~=~BLLiROTGXOkw2ZAi>TSmo*}%dO-VNl2?P-c3j>_{WjXLjZ#mEHk`z16H@m))`u=Thxx-Tk1GAnom+|>s!sk&M|gp;CUrDw!|fFxo@9zo>8?#ohcN<MOV=$!__Jw7>3&GlLVP-XBs?~DC6M-?TZs_Uu@A<n}c$b5f@m}gM><pJ)?2S4m1i~MTp%)^IJ6oer6Tj@a%z7()nsbb!16Ub>DV(GIfL55w9g%hY4E6lf?r4dSiY)F<+IwN<XSSv!Z8Ve(*akTx1V!#Z=sE$W5OdV5C0^N$KR8fF~anctVX!4}p%C>FRu*M(eX^JuuE8al`PKfY<`j7fz%72xE;rIKTi|g)~qoNQa(z7Kl#~7-hgCyTBbLk<k|d+?Buf0y<i^D}C9OX7N>eMKK%3EPxwFY{egXHV~}|@6hQ{fU4MAfw9Mz<e+&-xdhlQ7Wox9ifk^+8!qa2yAB>LZV}Lp2j*<>c;R`p@#^ns%>vIsvmLjq8@P|nyS99eH&_^O(=|bs-~&oEIguy0z|6e~*PP%zQR_X6Q(@7$!S-)K+9g8y5_b5;c5lb%8dIT3aT2<0In^V6fC3a-K{?OBADBz%@AM3R<9#$x52jZNYr0Zu$UA}*em$TD-8Z!mLtV_tjdRZhIJ?X=FW-W@slFvpTdF(#4;4+^x$#5{wZGAe-q@`*bT)FqsH-_uyM<DyRX(q}&D91BL&R>vF6<0b62};>1K4B0LqhKmJ^jmDkHM2yXu>whW;UPcfO1XDyZTN0s%qA0t}T}P)p~caZURP2mmdA~Iu<g4QT*B}=$dg@f?B9Qf1$Q7Rz)VnlvZk3mE<qg!-y-$)&d(ZeUf_EQ$I$Uy1TE@4x2v?OXhYCWykYexA<n5Y(~kjZHWb19r>+}cC<QbX?3)-RlLe-z1}kVkT9KGY_+^+Q3fz!_R}c3(62&hDmzgv16tjavWy||zG-YDv9rdF-sys2H?=%UdPzYR_gwz|WbZyUTl6#wtF67_Lm=L&L=q&ctp+XKl2%)9rTokV#U=h~8*-{reHV)KWSPIqx>!G8W3(GV9cyEY28pX}3s2$1HCb?TO^RUmifw=q{LF<#7mMu<zcEsQjk%UIe4wuObn6rv>nq25#wl9pqN}Z|Ns?G4E7zAyRKp+r08>GKV1?lhrLe+?+X%DWU0n+`9H@i9on9WG<JR-&{@Bp^zUh=7@7hQ<T&cyS2MPxKkc@7!RU0cD_&K6(ZO>6K$<vCM0md9_5#Zw1Md@GGt5p^-nF+5pe+_AOx6{BvQqwyCyM3Df)H;0-)5FHTXPoeM&z8{Yz+M-o_94}=#Y5&7kK%o!?fgkuc4Hc?Zg><u-K5SP!O}E~!CeN+e(H9vA+3{(O}Slj_#|g*5ZW6RNP>f(*b!v+`JmAk?+2bk;I_zXQNf-R@Bw|6^vpe1x17Saw2GB0X$2BGG{x~D)OOj$C;(`|9S`6@FLXllOg3oX!f#G>G=Ksce4oPZc>sdM#C~X7HuDl7#om5P0i8Z&x9xN~D*T2rihhx2eLy!m-=q;i{X9amqQ3QJqZ<|N;kf_WJ{<X=#fx*61#Sm;5in=Nh5j*Lce?>wI^Pb+mlQN19SJ2sEnjpmSJ_31J!1^~)#mT&A36AVbT8_1P1kdj-o<{E{@8?i__#01gU9}WW}5qdY)mKP#>55HEbgC*36x!}ffef6qsHEyB2QKFI!wQY7l=9Auz3^1dMck@opJ~Z#BJ{kxik>Q=ut5^c?%uYc<37vJ7FAa7dy?+!>5PW95W?)sdbm-7nfbrgb1E+nO1scUah;TSpsu#qgv6<t=?mpa@fwMzUV_m1@&R%me)=CYtDEU<PUntG?6sJlKeJHHt$YJjp*_P5di-VOybRDzM1#Q5D|?9w=>M7*C(Uk&3n9oltFwbltcH$HbedFhP1S+%=J|Icw$02aYTrFVr?=v3$093|HDvEBul`P!3bB~#}iFF+uCq%YafQKO^upOq2o8K(!a5<;72sn=ng-lcc;qz5YhM`_WO^c{Wy%;aFBy9JNy}Sc!bUS7~KzJG!k>qs=!J0?M#z-wyaas1)I5|71(#2JaBq3u%a(u?`&%LTeSP6Ri9CkM#}i9W#7uNLs+RRa5Ue)ViA<7Y+8-|Ga|1QGrpG`hEN0gGI|IoKJ?<KQot0<(V&8SYih(jTeXpeMo%1PHIk7&{%RX+w_p7+6%LsChX&HQ%Lsad?H$+_p5srJ7N{1S3=JUo-FBYW4w}hDy1mGfS@}NQ<QP_1;q3?KY$6&snvfkLMtx#?-9EvNEbOSpz>&)Snw@Ge|Co%Dw@TLztK#TVg^4N!y-2sqngUime)QSH)8SD1^PvqMd>ZggcDKquDrv{_ML>}rcbr4Y8t~j5t<}v?JWMf;fU$k>PGo>iK;nu&`Q~$JE&L9-Kcj*?yksMd9(lf3M2|Za>t$;%5A_?f)B?BX8;ej|J57w*mWxJcFCwu*I)D(IZp1kNS<*p=5F{)H^Q&4;8+*<P&}w6idBNB3w5se?vVRhz1WPMSswb8Yr%iZI4g=&hnyssRsbZO`-^U|oRo(f{y-eAF{|_2L;ipjiabQ*a01xX#DO9V5Nx;2(p}Rz`<;OYT3r)<|w)=`z)kbeZ7XTR;>@^nJd6%bP@34buRs7I0@i4N64*Rs|4U*hC19(>oW1UZ>FV+21s7sLl@6*;J#%<f|H5YpcTL4jdx7PbbI}InotIip$y8+a6m*5nJ_J(iA#UPh828#`>4g#_u7sn6?VO--Qb=I_YDCXH!FQRn2p1Gr6neXgEcfRi6EIO)=aumIxHg1Wg)J{`_?~KK+`W}7Qn1H@FhJqZ(4!J*6^TQp3gKN`dXrHL^>aa8I?JO{&E(-M9CaVv${qx9D%>l@&&!+R5J8?8f@2ZN0P%YW{J!;<Up1T!XY@5Y_36(2#isD_ZFgg^JPrK(J0Kd<c-5A9mIK`cwc;>OxIf86Xx*muwD#OlW0|>TDYBu&LH0b~q`D(i&IwD5=en1ns>Ndq8{@yrdaAs4M3^8KmA9BF>7Qyw6psAyf(cytoM=AM8<pE+F$GMi07U?QqUf=PAG5T+F!btxF5%2sz#+k(lp)@O(n8cy$tNE;L-F{;_t~>FZ)_2>fd73e8U9)%NQ|J7^&iBa{Bw<uUNR>|(q!{uiOHwX58Rr8mJp%pFmQcIon(!!+Eyg&S7&}Xk!o5hxd{GD*+pnq#NLvc&C@M`h>X$^T4|YGLqS}s#CxxlcvJ4jG5L?@(8jHB|MC?OY+^xu=y{wKQThz26^8yh`@;2m^Z$F(kwgG4J!iR#7*BDCsI#wYnLGZfP_ej`Sj{!HeROzhVrh1?f?Kf}CTK<5e^z&lgugiYMS87GIOgU8VvmWgvX>^elIK6OfDj)3G_8k6hN)D_<`m9enSUcQ^*kFhig*AW<_f(wW0Dk;G3~(<4{B(x#ACPqh<3_q?ZN=UiCGOo*qZs;h(>$x(0xaMoO<Oj>wUi~B4ncs$oue0p{JZT1u*ESY=M6o~ZDmHL%qUQXX^tmyX5GyahacK`oB|-cBlVe?QVo@bQyhJFWZ|J#`1VMIBgrpNs27}Aie@qxgyL!0obL%78?L5MWg_c^Ao#8MumhDa%NggL9k=cbx-)!k>5B&Wj1<T6gr-XiUWV~Cwih7`Rljru4#!1OVdeYSc+I$6k>lx6I}X;W0laC$`k0AL&m&G5<COM|)3J(9W2tAT@ImS|W>i2IAu#Lf-@XT<7>AioI2;-fOHVx{8X~E|MJ(H&Hn<%0zG!yDEV=4`nI*sj;1B-#h`&DKuPYJL7K{7y65|r#_^&a`vYnPE<YmW<5Vp~Wz35S_tTQGf4(~Yi?Z*^+e@-~f$|=EKu7Bvf@JCvg8x-?=g?YqBte@Afm8HyRic{!x=P2u+O(M9hdflSI7q7xIK+z~T{JTREm#C%!HlH*&0I;^kJ5OP%<O@CjWYt-?0{4UjvP4fmTSxXY6g1dBV`6M@_YR&*aMaba&>gmac=nAeANZC`+uF0G3|cpecC}@RU#-<*QPkz~zR_a5^Cny6m7(0!YCzVT^6EOF%tM5VqETL~w{>EL&%3K~^wJlQ8uy)&J9szXhswPsH0;RLwBV&bl3#qZIR{RReTx%;U$x0qyfg(Tqdiqd%_^sRs+=~e40pCN47D;m-BYF6%4lbmQJ{(rlWgtikcM9&C^Wb<??(7;c#6s~^X`=1jbb){-F?%l+93hC)kwDcfFih%m)bhNcW)%mmTgn~_h=<0s)sJ2!Ox;G+e+gUtW!PxL8spR%FfgOw8~B)8UHau;YhT&OVI9nFzo$`O|$7B3PP#rqY^T`Ak%{RAJkX><~+Q?`<i^S+e%ShOYuxE|6UXulR4>IAgJE?wbiu?>AIkjuHcoTyB1wWL2_h(Bat=0hUH;WJ?%a{lL-QB=-=tVSp+bOB`kqu#F?cjfiK12Frma*R^C<^;pSRI$WgZxvuso63mDAWCKY<>UMM})OQC3nCkAQrsj<8Uap5mxUVE-zM$_)^DSPr?rtdQqX^jOBC5w+wQOmh3H{!?C35i})$UhA3au$eHq;qY!X5(K)>do86n|CHmYbXaG?H1I`FU+V!Fh8{K(a5|FB|>RNt<)j@v`^7@4WI-N{}<cLCQwNAXSyHcn~?|85J>@VCPd)}%57pxjG8G^f*m??+`Riral*L#5{=uEhhV3e1QbrPB|XS@y40Id0F)|3JwD>BD^4k86bqeF0+_J3=R?fbt;F9;XU)BIl9XS1CfCV#s3p|NN8XHjCV>fN_vq-Hlkoht<`W6;l!-B77YU|dyE_w`>9?#d#*<Y+nG>K3!$+JpGssa?-pk-N+bwv-?%ZFULPz#|qC6ld`dvg};Xb5mO1Xuxe)^+Wt3^zm{@5;2vXKoexsXdlp<GL(cal6>nB1Id%S$nv)-EpFi(WaOfQ-Od_Y(21(Gy^}`W&c2T;`Qzr4({`O=^oCC4DU-!bX|S*=?&rru=Gf<g%|UFOaWv^0Zf^vw8o^BA?AWfNn1WW?ZgeN5vs|v)P&qs}xzs(SRocpY9G>4Y>YG)QoblF||7=hXcHZz-+Y~NEbJce;yC-|MU1%Zy)oFb~VgpAkuK{68%nx4~7r_HhT0hHhayUc@IlgX`itz46D_;pT=r~PzCvbb|fMrq4vR#mV^R9!0KkTp4e5|^cU>A(Pb@9<C??-cGo&>*7~Jc%QuEqe8<s32t`slN^sRS-q$VfDM>g^VM-gzQ`XaTIB6JGDt2h=7=#ue%v`C`@o{eqid({!xk8F0LP}E)Xe^h|+g|Y&7Z5KWNoLeD9Sr-sLt%dxfGu<Z-Y{T8&bzRt%*O@x5z0blRH@uDy>`@Rb)Y>jjIwNeFEt9bXtqQ_3n*d8+#f2mu#_Q4vrvFMvt;kw>OXDlF1qeM^}mA7y2krFRsp@g4F&R@`Ikd~P&eEJ>h{diarmG)YVcdz(M~;V>JQr{d&ij<cyI|1Rj2mKwG8ROdQUBmSUWcxR`L#;%5y$u<X27(csBsOLpBT&eV-(y518GqC_z#Z+0XM0houxrmMA=>&cEb`s~lOCBv~+`H6B7ai{9~w^Bt}ih6<ZeF)%jST8D^+NrYkUFx;DUTfHmmRAQz|2lEIo7sSab=s*ugAC$f6WD&r#X*ftJLtw^lF2vk_H95mmLmW5l2HVTOZIs8v?yAbGNlNFCd(o@OIhA!ve$6(c6Us6hlUe4HOnw1^IIXRXKkL`(Xa@&=+!|4UAJXpVMsQbaoz4oT=skH>wZ|Nt=!27Wp?3w~G0_RT%m;j_3+VSJQ*og22S}c5W^4d(S5!16Er{G5k2RA7DKmHjhg$rhKaBXpNPid^NeW#$3iRS_G2$es_ZdcCOW!zW%guajI4?}*RtPjyU#aHZdRDa~gptc~1^>5MgEYMoudS#(-o4hS^s+49ZBhEW;aC{2j1{U_`LBG700l!WeLDBS<n6w~q_1vSv?em;do+!uBe~PD#&&DJw(PZz6|JE;&>u$w{eQoP|6h{@iX!JH02&EEqW~b63FWZDU$@cA3;c+wBX|HNO#SXFXy&!G1a^g6Z@`gU@>MU~9VGcAPf>ltKhLM6$ittrsp*eYaF0k;iD~CH-B)ywR{Wgyuw3RhuXO;T4aPRL!UP{!UpeAczM$fGi;Cw$4%%E_hm2!}%}OU!X*fFR&$9~*9&N~aAb=qO_}gLi`vCxbhA@XcE-x;&d37nW`t&JD3f&yfFVxdzx`es_E{vi{tZ1Y~JV7DRNFa<U-Qp=5D+qj8Klr4oP`D@yovnRan44%`@qM<5<LJ#d_2yQq1YvwvaKOyhOAfKkp@7{{$5WO!m|rX54_CQ16l^SQR53kp)eS=MlL@mxJd(55NLS93wpFX`d9?;KLTc_1pXiueJ&9!BR)`ZX!?loC+v4~n%3<yw)?E3EPUTU@0$pZ3b-5g)f49Rm7y_xd7y>UKZ)}=1-i-FQISRC?<x~^B{?2-jXQ<mCwT8-Nf?nveY{|w0H1#Vqg{d!9L<Eyxf!(&@l@U;Fx-Ca}j=uY+AY7abMnn4N;iy&Wvmp9o`@nH}xCBhd2KbZ56g~C<$B?DJ!`3D+uYf)(DKXZN&wGyxgMF6P!b1g!RimJp!D=!n9=Z89%ve@=(WMG{`5(E-Hk^W-3?p28nWb~K+U(mpP=rcTdIngf!^bjY5otP`$v8y%ll}QO47IeJN>(cXxvw#Dx+Li+YnV1=H5=C_uhCFwU48&A#kSQV$O6yU9aq7I9%wrVuxJgH7hU9Cy+}RqSr{E{R13<n=Y(8*I6i(GgG<r5Sj~3E5|^t)+K@>uZ9|Sdm0hw0A>g%IRVP5oa=EQ7ljN?u045;kd;Gzq?EIu->^IPOU&U{rv=S5+31eUNO|o3h5Xzd;&+*`@oS4&PP64Vhr3Bu<9i$)~J{Ue24NsMR6f>f|fU>|t9vwO6zfA=_7Uln`X{plu2Fe1sX7Tqx-W7_i7-CmVhjeLT|5C#NgJCyLF9V4Mu9UB}xMLDq2}Y!Sm0<Yw?!>PW3{;}N7)BSgNi#j}+oYst_C)50mCELrj=4b4k&uQ{AKUcs{?g&xuYZ1cM>)+LC#`+G&Wl$n;QkAq#e+ufZ9i^`)l$$riFaH8o6vmw^3uwDv^ou~&OYm9hxL-piX5{dwd{*#5SII6Ap8Cdvgdcn9qZ&Y59f_NYBNRF=U19^n9e<<eD|{xifjd(*e@|FzED{;^!nkH8akr}0di7mY5;}}XaIWU1ObTI&_KL#o^?y?*B`rm3Nn-a4o(xdpHex8GiwI}X(5+;t{j2|n`ZREJ}EBX_O0iUpZEiJj!SE=F8eiZCIC2+Yq`qm?g8srYdz^HnJr67z41p~{3xvop7O4gn-xOAW4@3}3(Yd1oH}F`hB4gAgDAuO%Q(a{!aW%7IE<ke7ZAc_^(ZPhFNBa0v?O+CHB@h#<{jZbL^45Q#^R2a$sIe0HTpowKBJdNlM=3wqxdqM3^$u>nbL9ZFd0Y3$J3GVx)hp^!qbpunE2A;*yN&-f0Y$Dz|7K>lhlTwHj>sDA8F6U-QcY4@dzL^sdnC}HP{r!>0v}$=7y4gJ>k*2>(l5t#^{bnnxYxl=GtV#IcvSIyF=#?Y%Gtpe~rd}<bMDUcm^h{P0yE<I;^jTY_lzjZ1ab=r{`p`$S<JVMZSRMyR*f`nIn)qqXORCRGEL(<V>%Dyg1ltHHRVyBbHw#o%Q8)MaGDjNg)@B6ey;^GUtW|t;IZWjU)mtn3fy^raGn{TW0Swzn*>XRS;Q4-5TP9GpAKeP2)>ATqRoZ_`7J0(H}P{>x!<8F0%(qVYb}T@w+UR+Jq3DhiNiZDFE;DP^-;MT-xc;3rq!v4sd#1cRO${TU>OQx$Nz8onGwhC3KMcb<P@#Mdt>|icggPrh}|%iE6;k055pgD%#*IU9D5T>gsd`Z_B*s&MtAMnq3a^DqR++nuzdnj&hlH`ib(s>Li?xf!|Y}4+RRw2Uh#(NO8-{a#M4dSs-8njPkO(V;GT93{_Qq=7+8*(6<S8=m!7XCNHRmFK&~UHT`;7-zEf^o^%^$C!;y$kU%tUJKAn4D`w?9OI84Ap4Mqsk^C(BAziOgW#O4cp~6gCMAa}KphIt~vKY*_t98{Sp#q<iDqUn67|PnvX!0(*uE=_+r}Lqg8xZq4JsgKKOtFzJ(Q>#*3mJ*k8c3l%!1)&Qz;&_h0VCh-Z0kk;aYvq42u*DymEe7|>?Cl)vu3AuEfR$Tbrrkh@Ooi3Zkis{<@t4uPV@tA-C(J)g5Iw{>MGVBDO2lN^b`E;!M4Vk7u?BADRjqFq-*owuE_x*#r#l%)w_AV>GFf7at3>Fos{pC$rz0pSx8Yuylzi0iwHp+P(G3SQD*=})S93Rd3YRvlvs2-AI>{5=Ayq$i}^A;lLl7E4nC58dXrUEHt%v7(H*t}tjy{Iurj5u_HejNM-Lw*i+q`>eySlPEej>tN$4-?pUSe<R0FPXhSVae;VWm@h-_FK56`fQp!Ds)LZlh1SHy=6iiP%LSKURLwwPOlIS_wTYNNBzDV$4y+<k$BGRw@Fa;k~TZSgMIz~tG?rD`x#Lsil4!--BA_v5-I`wO>{0mX^(dPjS<hiBz#mDjKhWE5HRLm64}z$v=KgtM?c6lzM7Jz+Y3PnV&X?^F$#Rf7RB^<<_|QR*H%{yR)w9cUD~>V8hsZU9?Ow<G3-F4rL!V>}ac^<~L{S{0Okiqu`Wm{GjC2(RD-xj`Ve16ZpaCD1Y_al+AcC=hZAvf-;Izb3DrefRu_C(>Rgy}+|TdLeCdXx6Y_c4MP#luLMZ;JM(?Jw#r*AOV0K?1U9SBLD+Vx4dVQ4Kj%>^el^J#RWwb)6J|(%;d+-$a1)4Bz|Ff*;WIQ$T#HWP}wTZ+hsX>*Q}^6istCBL8eEddgEK5-EDzBE!Y<SvUE}_e(*GQZ{TR6LWP@q_ehM@uVQq&4$Ry$Gfi(D3ogM4pbX23@}+FnV*=PZWe1{KbPXU<0+|SPU9TESIybkmNe`qCDSTmdfl+qHatjm5Go5ACI|_K6jt#wR7z_RCXVDkE%*G8dMgNE$dl=?3cUhFUu952Sk9^x44`(=Z4{_nT-KAYrLele#X(p*yVf_3$A!!Mvr*7LGN4p!$81>HCFEt(1<^_kl`L4aIb>E`k+aTMEErVpar~DdEC@7sD1rxFAZOde2J=?+NRWCHpZSS<Z6BxfEjAbBU3`odvbXIIv@D0ciFU$HHyp7hT1%<P9>G_h@(TxPDQ*7D!B;-D)ZK5}dqFtJxu)~h6m?u-B!tO`gm70w)YQF4*^ihAs{6#rU1u$kX(JHT2n7Tn9VcK>y7$uGOL{8tSK~l<H9vH&a)K(LEdN9h0S$L?jsppz#>Nlv50qcDBZkef*0Lq;>(a##p0Vk0!;Nzc3M?ZF^O_xw1zH?#0o&;Sqh+U(HtkNxqTRjzW=kUQa<93cwBJ${_3~h9C6T8VwC&3+MN_ddmWQNdwI~*e|J$d+OeHrflcRAaJ>FbIngt=_sU~Frn&*<y+Bg82{O&0-I4j}{_DWo~n0r7>ZH6@sXv1e$9Asmv0sQor^=wK`)fIb2^454-zi_mHlvksq#1fXvLw4vn+rqLIz>qbt!Dw)GB#|MbrC>cpOq6YA)AH*iiYH^uGAkmxymo7;^zap~-CE;&3e8u7~A9jWvM_tz6&_+FY;s5hMIpXO)63<fJ?{cm18<?wHfK(G$z+U7&dVrsCFGUBqiinwUaF2KZ?03(IEIMeZw#?^pJ7v~dVQ`n|Mxb_kd?WC<wL_$24x}Aof&#{wpR61=F<;tN^`tX7O2KqF;4Kd&viDGyHN{a6g|zOhuj@;WfkZMW{-|a-|5$@SitbGKTQ%5mEI`xZx~m(e)&MLsea76r6OJa>bR5c#(!LJ)!?7loH`85G!cZm`>n$0pIJk9YTj*8;J@xud2tC`P$2ouXvGJr*okl@wR?upQ&Mu#zSfRTNI&<<ftCl<^iqURpX>_W!GOQEXp$bvT8J#b&X^^Fwoz1d!jp~iI5;~^Fp3jNCc@pK$%W}C#B{Kt&mSO~DMb{#0mX-R9hQ>Qw@m1m-8WeO?%~osJCDF=_Mn?^31*r1AzNyK73PA7H)<{jIuW~~_;Ts;rlSlL0{tbespH{VE&Y{HDU6JCDgDelTIk(1tsJ@19S6UDH?Wh-^Gv-^l{*S1?`LCyxG1Eo&+17Su((hJ~c=Jl0<n4Wzom`8K&azA)EIT4QGl2}GXEseSThmQ78yalabG+R*jomadT`*7JR`of@vWkG2uovOPvN#upV14_6Xg249&~Fc~Z`foQkohbiA{R}3c;-BK7VT3gS(@8(d|25~F6W6+h&8V-OcpYE;N5x3r6w9#C8`u8ASJk<L69yrz_u!Ku)x>)#;sXb1$rI}NECv2xolcQkzEClIDzGxF6C91ZkC;Bd(QMySKI0fRi&#ud->$2$xuR(zUB@4zbl=O=)_)eWzm^uHAWRE5Zf79KG3cf(e2Z<5nEXGqxE(&wYnKnJH9Nj!!VsSOL>=hc<<#5Mbj_^i%~S$1C@rqYyY_$m(<2Z@3T$5xV93T);5PFw>#Zb-RrsMAG*IYsz(&Yl*Z93=WKp$HTGO*{X*S}QgJv#v1f_yP8(XNX!Z&71Iob}=OdaxV63I`9Ft*^^@HNj`a(+>C$Wv1A+r)<NO!7dJrjzIKNZQ2L{4zB5g<P_`Z!TzE>u`bCUeWc&`+H;!=S{e5D$D*^$R7>?U(wEGzfD18VCs!Vo5(XMqDf|GO=nx!XGLhHJ&u-Jv@Du$=VCmuZFg<b;Vm;J&og8%L+f7BfEOXPuD%(L8sV^&bp`5o<_K>(?JxBy0R<RV~bidzB6i4jkyOesiRTkq1n$_@<@S1)0$E&HH*R`kR{=4aaUYvLLZ84m@7$j25qr-&~ZB4?ap+zR$QMn4gT*aKM(ATQi6U`M}CJ=^w4P{0DT&czgtgwC|_vkOaF(Y|4&GYvBHe_)lgKn3TTZ!yKu~2xev^pd#OS7mAG=~kT{YLutHa>Yc^0%KyrHJMn7MMI_=aysBeZgrY>r)fZNZ}QuITX6;}CGdHoV+6&f-+b2VF4(e?p7yZzG+;9oY<5lO|*Jelgvib60p<gx>YYY7aqw)Y@}GHD*##!JYo*ti?kP#q<7ncmJ5gfkYm(h|X(94g0H@4%BnI2x*z<6of8CB>-u^NddaAf)ye7!ZL}YT7-4b>RpyB9GC6?9%3N2Cx>(An>1wrYQK2Q6JlUeuvBXYA3&=_^%`&U-FkS!X1Tn+_RO~{y$>+e>yRJzc?J6E{90Rfz#ukD=hz)ku~MgZr$J9^}q83(A@3z9vB4l!}dPs^7-(6kG5T}50mz-U>cNFsQ!N`IR$^W$SAtA|7qltu&}{b5#zAn608BRsgfhr1egql+z9ZjFdBb`+~awV52~zIVHmn}3o`-9Oa<}ayP%N7)TVpW0u<dRk+nsEewY|mY5%nkVxzB@v=p^15(@E45>0pi4LQKTWG8cscW70!_q2w#+G1V@yc~f}(9YRYa?DjyWFM3QLOKpWPY+hC!XS~8fUA6GVGN}(3YoNduQTUTWG9C)sp!HixEbLLjbF&vggjckZfY^czVS;O>KSu@VX6hX(>#^w%w0azQuR4~<N58b?GA{vz*Jx=V4Q!B=%39}_~BZu@=OQe2u1H}?qT&g1|U%gQD1RmO(PiVNO-ArJm-2m*Ctg)Q$3x}*`<eGurp+a(zf(1>qTf0gh8B_`9~Vdvm2K{w`5esj(N(GMMd>fCdTZoNtq1Lgr#SZz@NY;&=GpOz8+&a`9cLTQ4=Il&=X~$f?$nxvo(=$+Tt@B3|zp9hAgiznbz>ZZiBET`an`VZrF)<VX84By+#F8*Er6jXFq@a<d_U;T+<>Qa3&bQBhVG=N7PuAuBk(fLAKK+o41`sC%xvU1xDb?KhiC$U=2<gd;Oby$cuUT0m@FM4MT@2)K1vn;ly>ASeF(0@&g*i#hS-Te2L}VyXPfEC%52f(nh#yI7fL0KlX}}$F_W-d1u4=xMwViecOA7S%{GvH_X?JN0YkOCs&b>siI6MC;P4ahdsg3aT}qX&DN*8y_{C~?rT1=SL*J#Dd1$4UNCVTGJ&zn4q6I2vhMg3tYl*UEUo)#1(3hEZe$a8&Q9-U45y()^D<+#KZ_w~1sb|u>^QlPw69I8Hc(lpxaNzd(hZ*3(ZX<_?TT&62pBY{Ee7PtZ05UQMVw%GO3a#>qsW&;Q(s$M=n9-PsR74X!X^rn9R+vs3n>$`Ht8VgAX%7rAR6$+rrfUiZPj(g-`0%2j7C|XY+fx0es`zg=C#h5^A>3ignRunW~NY_9oUHpP3dz=<UibJ89&8!W^8A(w0doTK1@D{0VFZLwFx5R&T5C_qd^ApQT>wYS56Kckc0U(MI%##-8+2ySD-`tzQ9)I$MZX>#cn;YRISTN%20=e@ieZ+vg>H$7&rN--+44t6t(WVF;%<XH;gicom-U!MnX?mSjDCykMd9$-!}>CYoXFY45<@=3}ZYt2;l|b4zHklD%lP2&hE+J0cB(HTk11z97oN*KM20Sa<;@w>Y-8;b`o=^4n`*))aLL4WzvniLPNBFc;eZvN*qNJx9+ZiZ>}H{WTQ#$0344S8EOQn(AacZSD~6(LKL#It5tQ`@JMe+77DAL=|Iohmero&j?i7jj=h}=t_3Sx!>O|y=66=~?6K%6tqkru^cENn8=iUYJWoJSRDldHGgK_%@aI<;)H0pt+bYrTcSo3E9%*v3C21r03~)O%^gH0@K^0Am4W|-y*{a;iwz{0{*PepbQvIM}!eX7uleK!YxU^$x6|`iKM`hDw9CnOzk*k;pyNb9lR=JK5NpE_)k-$KK1!r>=?db4+oQ#2F(w&>F2ec<qa*q2onP5()P7FI03<a;Gy;gX;HD4RsHZfWqAC<n^pgRoeUKE4@-S%7cB73{?Ktp||1%$?jg9Qel-h4Z2(-pYAY6Qr0Wj3H{Sa^uN-9}PIZ8xKbciwKSy|Mmf1$kXlePQjLRtR)8<8aVYb%Q!+E9)O%qV8d1m5Gj?T7mq_?}NEtY&QCA)u7g9&$k;6Dy8meJF%Y%J<z;v*KJm@n_W6wp~2=pimk)vss0=?AG(2Bmc#05u2e^}(n}iP%s&<d3}bB*1nP8~`~x?ja=S@HN}a?cS?xM-^cHpaHstqAgYwDIn%J07?77_`>?q0Y?ncapqFZyV#a(mjU09yT?6L>7BtlHN!hy@peU2)R51%YB+9|)>(!~<_a`_W7M#W=LU{qJU?b{kmB(Vp{Nk}bv%a^at#$gw|L$p9jPv3m$NCd5)n;)2Iw8JwrYA<3}XnPssjIy_c^)lJ96**G3o@e|RoSas`Ywe#aE`<BnUPfcTzeFN}bhZ>dT&nvIdWvD8e!E(w$Z8LF1yp7Su=0@iLaFC3zy1E@^W>N3uU>!u<I9HAV4JS9qjPmAH;tXk*T};EiQ`56w+aHK_4?ruP-ar^azvD|mv#hz3h3NhVhV|iQgXxf1nq%Z3>*Qkr`LuXl8@j0DdA~<EGQ2+a-DM=u56B%XDKWaa=O9$Hx)-0zAW;ZEZ6>|u(gawLU<%_uB|a=;TjpSzIZ3vvB7~fUc@W%9^+5DJ3FY8s~N~jm3qWO5dGYDi;gt8%wcA1W|t^yal6L^AAQpcL$Pahk!~H6%$+%g1TY9iUqr)*EDsqYeBR|6E^4yzq^d@_+t0IzGdI51jz7DyT97kZhVhiOYki?_>FggQpuC`CLeD`AamrFLTWVX8E0&YEMp=X8g*R9>R3(#<_b~n%N5{wXXuIt?A99s1qEYC~Bk9JY!S)UXbh<+K!X<30H3v;p%%4X_+VF5!%#no~yyqhon{FmnL!bvodZYpqRj+rDkK7l5#2D~);)iv|#35)|Z~;3W#@0va)S)Pbbj6G!%?~2gn;1vkH)_#KcFFsZR(wzim(uUD3>7;}rHLr-iYaDnsXPp-rJX0Fl~q=#^eD2I3_$gaA3VD;kb`Ev21oaUTc@dq@S7G*QhZ|In3W=oqiAiwQJ1?s-YiF(fDDc45tsy;_6nXj@!)FIUqywu)vMjgNuA=n4=0JMJ~N}fVVM2*!;TY5j|>V7ACQ~A!Ce&)opE-0p5dyHnec9kTRQ{MB*CB0P627Jz;jAe<rfXuG>MI5M^7?7JX02Tc_e8dHh{H}_Ci{OyoBAb*DyZI>qPBN-fkqfA+0s;5fUwtWh28F%ub~0ogKL|9Dz(s&!!VWd$q)g3Z|r_Q>Zj*o;0(l=oTIy^_+~Dc@uwVe%%?zgxD>|Jn=5!)jo;%b^K7$(lXDcY?T502~p~Z;5hL#mW4Lu2bu9=>N7N;clkl5801-_NgRu5$TopGl%<Mxo{?o2r=XxT5H<$AnE@DJ!VhU<=fMFHZ;tjkS-hfD&0E;D(4^tX@c#YieLCA>_N(nVrc)%tJ@_AfuCgUtNRkMb4fw$I44tRZIySnOE}#SVcuJFm7rh0c`5g+-$A*I_Fqi``0tr#Yo-an@6K(T^dSi_Hg_dk(L1~+w43!lLX|z8#@7-iwP-Bxt+d}Cg2fCL>HVKlk-4seg%w)X2W?R94l2Q+Zt8j%0-+40Z(0pX;PJbHzOLna$F<`IK{V1muAeK7I%=INip-5|Hftu6lFfoa)(F72dHs^1J4GlwjI~iPgN=y*3slrpKiO?J_zoMlt_&p_t^s=K%&P!<@yDD3{{P0G9cw;}<B)f`Yc+^*|0+t>kO_eVpfA`L+#->6Mj#bn=(hGHe^|Qv7&#1B;2O}M_Am6;B7@`Xx`g)yQB{y97HYr9YNhf5bO>x-(p8;+26}rH58BU9#G)#{=FSbn?qS`JH!!d!IAWv0rbPgA%Feg4EN*H|A%NvGt+r(E5o2g~G*Ivb>wt_{$a}krG=!rZLcuXl*N;|HoYB#!8l|YIUXT~jvs#f)=V#8uyO+;^??XzIpetmByQ6<dfDRI1~nm=;0iz#L=&wJwRd0;A^a<Qngy3(m}INvCmW;{g844prn_A?Y+G2qU2x$JfyW6~_e#{$uKXo_3wbZ%s7&T$;>ruE1?0qE*~Q_AD@-~^L&4M(RBADgT|g_qZK(iC<WZx(8d_!6r@Rt|Oa&Yp>&rC2E<mFq-OWh*D~Xbl1{cL+K@FBjus+(`7M3g|l=W29&|s+|x|_Cjn1g=v`w)m$bV!lsCJ26acORy}4Z1p1AdG)o1koYNhGy%Mj2R~c%LxUptu9f=&1S-rPusxbG_F7!lVB?}PA)7FelZxzm>WU_ZOC0NQ=pf}4kY?LL4ThVvukWl9^Yo8_g+wQ#Pd7sbK3`&<|+{M8$l^)Q>cQLV(#8g%3oQ|aPOEm!5ZiUk8gIyx4+T%%`a+$s-71%n9CXO>iPiqHzh!vd9in@pT#;FeWO!6Tc@G^RRjRaejQ3|~i3>Vs?6ER>`33rErbnZpDhH)ye2~7EjcU=lDJMyrgPw_)#_|W0#G3`?rJ5;@b6Gm6pjoF^m@{%Jb$t1;&=~CiS)qY7?YncNH`%xU5d2JUqh}exhi6XzDZeC|N?9XfEZ*%ITx}40(#04<{*0No7J4HFGd&PQI_rD}BNd>!SsR3j2whK6xYZwI753n$J9CeY9z}|_5_-`GH-U1Is!#qk;$?irU(lsXZq~wON0_z`6D4|`xpu}$(R#s`SPk^y$Z6z8MBp2EHwK@!#mEvT!QS=86T%XXH4{?HrxqcX;5d*FJ8Tvp|d3>GqAMZaKq{X}bdb2H1EWKVaoao!<zx?!hl;sy$p(ZBE+{#HiA~y+`_ltZ}S^w*2ZJFWJTtBg-d^V|1ra(;kcksrcU<M7sbjsdmGqqY(T5-VUvO7~dp;9MN;pj`s3CYZh-cUZb*k+Bl)QQR3(ed;_oQvx&LTnsTZR;=c;>q$tEq)}AA~_vl=#&CO58Rq8#ZBnV50@YS7ko)bJwj0fTbjbGLq3bU3ZXoqxADYD&IiQa&S-fUK!#e;S5eoJbFRoa7v#hurk?VeSmE|NPXLuBZB=V(>QmBm9?{JOrhWohdKFL3&iX^`g0Ui7nbm~K^2_%Wyrz;E0~V@D?7S$eayskf&w&*u3WDq>^lJ+LXxPY61Vedd+!f(C?xZXXWD?|wASO2MlQiSx*2n`%8SLF~g{Voufl^7%oog}&CQ{mdQ<%eh^~=3pVk+?c690C0P-W$XP8fJu#guJj8C;iBMk_}ys;9ZdsHelJ202D=FU$Cq+Q+Ml)LbK29Qzbg_}dL`j<nq?4Z@pCIuB#cP~;}(IY<hqY2Vf$QT68;DljvOn~wtbu?o;~o>pWoC^}Q|iT``6D(WFFd0l18MUUe{MD-<f2vrVDEM#u%m$=$4>l{`1c;0%SAtM96%!lv@WMI6j;e*(AW&Dy+t{{jYduXG!=wDu-g<U=e(Rz)S%2i#tx`;gt@nS;@$gE+UTg_36d11GGyc6@p7XKr#&(K`i$wn#pDhBfYG&XAHfaEg<W`NV!_I?MiL2CagoW@|M)6(K^$8bPTG$^3^?sAlb&uDBAZ8&u|#7%4yy9M?q2ocs09!vYII~03>rKpb0hzf?fW(A_Y<T!BU{OW!-u+6-DW@PwNT)KMHhUCz$pGMQ>w_nZp?o?MBeCTd9)JtRJhhAk?o!wAjtI7ClU=sd%*yPiqBw0NGLjn$lc0LEH9q^<@CDDr&cv#;+h%h_ACMVORvY_bVS1HEdF&rO{eH-AHt8$(#6_<Ykea)-uh?5cMc_0XecdG14VDF&uik0tOwZ<3uf(jR30%u26Dlzj-iB_j{55m;RD3UE}rWrrLJxe!zU^?0PCN;TG2e1M0Nr=!@i5_xPKQpeOj<Gcr=77LiO&n=JU4)@6F1D~w=|Djj8&0jlN@>Pc?Kr%Jox_yJZqo;D0O1<e#5!(o2Z1k}i`+l8g4lP8$%hA@p)zJLJc$E}AKuZ>2&3N4eF$c5ce|u(IvI>e;e?tbgpO|><DcO}eAB1z@I+2Rk{9{S%1mC*B)hfOGXf_{J3i0T3sgCv4^-cD_+C_#{Ax=N-uUa`ar^ZPTekz<6CRM(?nBQ=%XadB{*IUcL!ik>DdC8`XqigoM)2}6?0|>TfK;#E&f_5Vl^S{YA>XUss&CNcXWo93QVi*kT==Yq2SO0#hNE~QHv`@b&)X|1WOvv9YxJw(cWB*J0cX1geuo)dStew2dsuPL;53crM&>N2Sv5i)uK=Fyq4f4)wr82Lu7i?_>xS&+W@q!<V3uz>(>M|m?#`Fp6xc7RyLOa0r!sNVYuQ9!(+fM%+(htZt#YG(&6d2$h@+s_Q<9@-T4k{FX&&675in{zE60oUZe|ZhyAhpt;=L6*ojq2!gzhxq4}&)c9ns78pCV++p8PRlrr33CCw2>(itea|Z^)TKzBJ+`vM9IU;EkLI3Lg6}331L-jgr@VbqVsAO81A_zwn6bLEJel_yj1ktkkQ#a30eu@o3>4a31u<2h#_usRiovIfoG1`%(0_)F${5gMp;SBcPG|kRR#I$~tKw_AwI?F{iw+ZF}Ce%j=Pnfua)wHac_RPQ@|_E`vKk(@3{A1T&<^WEi)yfSxx%$_N@~5D7C~()>#k3$tzZVA)k<;6X)%uQ<(5Q%-RM+JvdbJst|h4CMPR@bA9*8VgidEQ1x?M2FAGDF!UMhPn>YK1!M!s2X1(g-C_OFSKj^YJH;EKUcWM{edoPuDY;w1D?jSy3gE>Fve1PbJPEtxJpa9SX9d40R}(9A6?TiCsjfhQJ@Pt*af0<%urd4L7oDTk9y<<hH?X~plFEPOdZkZAoY!T#QP?TaXvElP>jf6u`SvTmeqDPV|!$-K~A}<n)+rUmM$stgGmmiN?o~ISI0~P15+0ydI|kmD%v$=hEvy&D<6@SRX%bKrDg`J!>$4X53)9p$9#j|jv+Rujxe8*!&C!oXHYG=)E-h7_7vDNkj!63Cwz&)<vRu15P7P#>jEV58n4XwzA}Y<<eKELgAY*8ud>Yy_aSKojSexSMkl^kKt1-8lxw-i**nBGfc(;?*xq52JNj(+c=ULvZejTrV5Gu;RaM}`@PH)v$B)%4T(LocD56gSrw>`)eL!am{{0zA!VezmIc)Etl@(cDX&)98vvA?SHfMQwfjc^}4d<OsPlX|$ELAGk9GJKcdJ$5;EKi}!hh#%RIXn)f(P?E~rhrVy;9+4r=UB3+OVm-+AG@YqLyK~mRqw9Vt-LC(-y-`)W{->pFO~qCYwo);33?w<lv}Uy>5*euE3U;QS1mtyW~{vD5<O7Fsk+?}(mi&Kb%o{e-yV6!D0xu;7Gw`5k1UH6)z-*{i;Ds4A-im0TXRQL0k)i_o_k(I8Z{N>f*<dTyLFg;Ff#c0-n~&wmeU#)2hd-mZ_{m6;We;P&+=t##gYb|u;<xqNo%qGP%25p2q%`z13$ygdYQR*ie=TS@IDd4wZf9=G2U^g-u<yd96U*lf$a~KDYCMrqoa)H`dnJmYA%Jsq>(UPOYAD(6N>=!Mtq1-2U;;S3~Q??rKWt0DuqYG$0w)ix|I|kW8d0R8+6DMDV8L+!)N4e>0kQe=jNqY_#|pU>9go(dNWiWuq-WUcUaQM!$gr(F|D{~*jR#-H&ALBWmmHt(FuH-Q`)3iod7GzFA9!w@9?UI6$?3tk!h{5fS$^@7-r5LYBOVMB9ZrMagD(RHRGmw67;bOWf@(J-Q8h~O>cX^I-u3J<in~kN|<-qLa2W*BO{5;cOEDwC7Kx^d2CITHAJ9J9*35q!>LU3)EpPROq`YGwEiUq1H%2+ZOOcL3RCcIQS7KS#Z?HINq-m&s~tYKHcZtzMo1Ct42}xuiH3}NSynY(f6=k4F8k<}uV<pgqSNB?Y9_iSnV$LyPXIN*v+V{Xdp##4apLBneG*KEz;PV9A~tEt^y|2F%{uwcywBEj4G%^JS8H;k&F$<#RdK-gwrMg8hwmJ`+}KGmFUV`b1Lf(SkE2oq#AO9M?jbFeM{b7{icPXMP}N03C^xF_LR}Yb?}9|G7ic)#C9^IT+TPp`p%q#=u~&JGLLfyQ#WPQn16bf=w3@W%VX@Ak4-<2SnubcEg3u{V$DS#OC+a?wL{a2Hn3Ecn!tD*Is$WL(={~8}kD6-3BbooPNQB#r3bp6&CC+7{-VWUZmVQ0)t{a`oW;bvvi%nujWh-FtG1Cl#w0NX3mS<7B9O;-!vw7&*R47WKj{=^kTHzmgnGYL{HF<#3sO9Ij^}phck;&v3!PZRCj~C1Q;<DzP*%gjzqUO>dioPkAbCy5VgzUS!#a|`ch8>OWwyF}2fbce(Uq|yTjVxyKHH#opS$=WugDeTsTfMxtqH9N>;Hm_I6P;$Lx1JoDE7tBobwKr;#)%iCwiwTG$Df|eu^rukT0*XN!Nd!~udZQg`zizwv5Qa=&^2Iv|Fy(1z<lH^YcQBSRhh0<k>SxhFvVSZhj~@FnxKDx^MqDY`*^sT99^M(Tn^-tp7FsR0g`XdJbNa?#@alk!bVL>d(GD&O`E2kRG$M<bf*%~s9R;vYm`<ZN5n#5aao=NKe<*;YbcCSx6rnT7=k^x&}EphiNY_lI_1mc=(}xkK~q#N{}nJP)I;*6a$VMRGflRKDk}`0sFs;>s77AJm+2Zowm~7oGrVbEp}1J$@i&UT7k6?P%AI9Pm><xn(k4NF5F{`mcvzDYG+nUIvkSbfWm!~r&h0d&d(~L5a3L|lta@p&Ma=<TvZsRiEujNT_bOaP;~q<+*=OWf;#j|W$Hc_iBgzA_-4eBBo;pq_@ibVAzcs1=utW_<TO<5gSI~pTrC|}djY>-p0sE_Fr*!+gj8F?8PF}zSm)jL7>5N93O(THKtmJa*D}>a&vYmK@AbFWInl?r=n2bhTh|m)90Zl?w(fGNpmmURW&ulR#utngIDnOWQr_H+}#fx?vOvH?6b<cFFk?&B`D_1fSbqR~`m2THD6F-vKo4t8w4MnH%#1VMT_?D@OOWgXw)E66sHc#)lIkOuM0*HLPJCMFTiHS>Cym{3bd82%lUclKG*wivh-=!u4tu>7@H*eV9ju9l`P>ObP`2MhM#OjFcD3cxjo}mW*^%EneHVubWDt?X3k~YQNM|gWP@TvSwg@2VZ0Z%YG6q)n>%W>4c?)~A%jpQnr`9i=P--w4>G`=dgb(PQAFIgH6I(PF?#{+rOj@PZ92IG&>V)+{k7i?Eg`NXv3_iBD2Lj@kJ71Ad_k51~KAm$K@BPt3M7BkbY6B?$-8K6$sw&f6Aj@!%Fnz%GHwqk)cm)%ZF^gBrenu+SIIA{a&jP8&f>lb?E8Cg49kM0_sJCFuRR#e&Qe5n`^Iv=6+4}IHVt)V9(m8<?lvyK*tnC~=q3BMuLl8f~&y^^9si%!X*Qd2*M-Rx2Cnx+uWou>&tL>KBpGtWCN<!f~4EUpvV^j)8blaAEbdI*>532}b%F66jtCOoMq=f%UxM@}KSg`e&U!s$U4SL8cDH%f)BoB3UvfeyQDj-CE<4QHlIgjz|_&z{I_Y)*paxQm<mbXv7Q_}n`69e&j@mEcT;yDR&H0&KF_ioT47O<Mw`>@7GdBF#A``UqJh5D}Q21<H%3i8~|Adlh`e^vgz9OVFV3obS&r$^~_<d(^ahMC!<9zZ_d=YNEtCxQlGKrN<5%RDgPFCg^}SXh!id(>Gc=PgPe&Z={S@GWrcB0h>D6&XlS`IqGBFLNxlsh!*_a57;JIX{BYfQ!Ub(@NMV?buxN%@_01z_$YZ(y1W!#J=kj|_h_F}mFNxH0Va(EMs7?Oldi2@?mWoLiw7yOBysb`?FnP|{3?U4P>42V+oH{5h^eB4aSjs5@q0YW!4ZD8DalyFDizeFHLE+GJ8Fq7@(mL~TE)i0_&RI=Otk7nPlm@fT8fNUqlbjOy+H`fmY97G@7>w(MqSn$6-Ezvam6+_KOB?I0S8bXM8-G3^Gvcg)vI*PKCtL;up(0(RG~zRO}R4CtG>>K7I?fbvm&~rJTrJjOOZ(=2|gE2FbD(nE9=Z&BY!BjP<IYZZpv#7F@lA(AyLl6SOj?^h11u}^BzzO+F5O}8v`%k*Usat7pxudKrp=KKzDW?Gj{vao3bWGVSzCKE>aPuko&?O5fhAeNhyV^E~O8ol*R)};Z<i>br)7si7{`-oo&77Kkg`-8_Ers7cLi?N?T*R8DOKj@`KLbs$#w>7%C!^L1)i#7=tG%n<?El3(*ZsA<|nrL45v8MPao;geVA;Ny>=}q)CWo5Fukq3?$NHggLZ=V{_}o>4ycOYg34_;Q{%sSG0O5BdLzlpje$*CZ@HHYY~9+Fvql$rqR1)0$+5zCax78fm~(jjGd|Uiq&Cmh0R`)Q=&Nz?N)x!F^fA>9s4au?NX48P6W6rGZEG1RcvCt*;l*QgVppZ^R{_qk|(NM38q~*fK3`vb`3J<6F?nllpvfKTBtjd;n{Q#mVWXB<>R7<-eY*KsG=i=scpV3XxO5&E#4L72OT&|2i1yhTyTHc+*%I>l0i_6xUL*|55vZ!GH&Aw`S7xqPpt`WI&K8&It9rHTWh2w=mf8x(>oMU61l``d%n!8OT2Ol!wD8>!XYUjVIJhX2VS4+v<sNnKz?k`z7Q<68wTz39!g4x;Yfg$OCItgPaJX183~c+wdcs-R-KLuxv_Bc`nG1m9M#dSHo@89st5zl<~js%+h;gc6!oq>zFCR4!(~oK^U1Imy{!8~9UZ4_*`7%bYZ!p;sof0zGH%ev-6>SMZFfl>kUNqWPZha|kC{DG4gSD<4a5|-B_#R8WM^<h7fx?XT9k<wFMMic7bhO2BgFKCKr)r#B@M6Wa0oyQ^he?SdKBKUhZC3N=Z<IJ&4CmrfbL|`(ZFZ}>TsM)6xWf>QTL{js<P?ZX6$Vw1#QLvx?>V0n9auld#Gir(}0ZTZ3B683~!wo3e8M);71uY@=htg)e*;u8|g#d83@!V>?TOC`-(U;HnR+=GB)t5HM2(=t00K(mNJbScWSPV=o2F~`mNmV#!?53lkxi~a33|KTWJzu+Q8n@N(Wu$9EjuH2M@LJT3d1*xD?4uQLZ{pmb2X!&So5aI-pIri=IxN#w+h&xMr!k$=CK9RZIMP0nI^EX|7Q_0JiMJfxsCZFVutI8_#&#LCde|V;iiih_J_wTqD;`lf4i$lC3>gPJgtUHVCl8YOno>{pDD~-YMLAw?FoVNG*rJbx@CD82C4JCV&C?I&~Uw)%{e7A%JfLcDKq_#J^<H%f?(6g#7l~@6AJF>|C8s^ksSTmR#P~*<fuU)4*LW|9ar>*n`HrYG94tSpu&>&<7oRTGElOIqY4C!OnXh^j-+$B{0#U^={d8Tu>XP>$h;=zUeiwbKBJAnkj|!9gko!Ev^#oMMu+bLN##+Gw=B}sO>CG`dft3o*VoHqx+p55W7Y@qD=<64^wBB1${1dn`gKQ%-m7yH5RgDrOh?l;@=>Ub>{xvux$)UuelSr=$*tZoK3I1<`m&iad@Ey2FfYVAQpLT=w`0wS}&)Y47cZ^x$mko!1ovf^>82xF3aqBoxcMTp}R$g8gcahxgQ-w@vHX=oru+08>N}bv5M1Y3V2fl!weYSYEq-n$Fx>~j`;dg@>kScpnpSJVdQYq7t|a{D#5!3Jpt8Ctc&Y}6RN2Fta-?P;rMrPe^i}l?9Lkf`j|0tOc@TwQA`PAj=UE&<A@*1Sn}lMlNbN``uo?-^lmPP#>i;Si)KlauWJ%MxR8!CSLq;faR|{IzX?)sBb_cqN*R7Wl6-Enk>vBLkyV9UK#WG6T|aaNl9s~1J5h_Dg~dHmH+l>sYe|hOU#h~);A0WN9YkwtMoDeonLEz{rRiQDEjtsZn#IDHq93=yqRHPd4iN4tu%6L6fEP(HNH>FMm?rFUd0}X-kBhn~aaDh3QFDoem4s6#Pfy@%hDj#Q#A2e7%>$a66#>1TD=z?Pfm(bCN)+peo6nxS?NoX&zVTGoii%}+JeX?eC}N@?6ES=~+ToZKw*b%HuN%O{fyA$+ChS2tH3g@=NNMmPtXjbDIuly(w@ywYkd@b1Sk0=gKbBQ>Oq0^w8hze=2aYZP9Kcn?Z&g9=RA{T7cLMBi*gG3E)y7s8;Zt+88Gydw4AJ2sjXp+S-*^0jn90!M+c?OuGq9CRI)EA(BU-;XXM2$rm4<s;4VvBNt->ih#gs>lvyrxpyDsYN-@WGO1o)g{Np`@cH4g2`lsd9X7dqZc>>K6O>3P1y#Gx0Pe13>cPNX#C@=GW;6_mVpZ&t2WWwFn;<khyo_`xvdI#vUD#vFfu5hy<#<HSA|CWnYA$9Ld}ju@yHwZ!TaX)L3im>5R#G97SLmbzCm@gCXZkTzKN1p7o8X;pD?RyD#(o4u2#pTBoc!H<tZ;U8`~stIBOgm*_(P`CJEY&E1F-5&uOv}@px+ji5u3EeHI@_0YAYfqiZD;UIc8Zc)I#3yRzm(vrjkHhF?m(zoQ=$B4PHbRQGSrGm`Uo{Lf(~aYn;^|eRIE9-F7pH|Dtboy~n#~zcHnzo4Q1}7}*9I@CWwxkWQ@EiMY53#5h4Zy(hKz8Sac7m@Tl?qNef5)4Ghtd$G7?gTJxWJFfwXbenD|}X2@_HzjNEQ#2RG%Qv7Tx0-$vAgyn;ncA$fE%y`FA9-j}gc*+W_Co3vYzYS?JD6l{rrNeiOBF~-5=&<Ir)QtaszbRa*Plp^}VMw(O72QNJj<1%eg><Q%_&McmE6iUpF8-7sB5+_)&41I~MHHQYpJ`{9oYU}dLeb!Abm8qGEv!duxsX0H2rhMp}ZZYu>=>i`e4`5-n@3x*Y;)OLUYN(gGzqTr04}B>ufOJ$*$3WQDu*ZNSU*+ZnG{D!hGFx;T$n^ra#S?T|5XRahx52mM(PJhO2t&xJ^JV6s(;Ys-pk0q2V0gJ^?Y}+TU3-MdrT+FAMz9Oq|GC5v=D%(I_KGw*VM2{*bqW8w3-jGotcslxKw|IP(qiMEZ09r%$i8|zgEi=?G}^FHY+++Tp!P5;ZCH5EXr4X2(6zdtmQ3+9**IO=@l;{&yy@K%{!YRd<HTnV{j{U#-yX`Me|y+W^_qwoLb>kEQxk0K8QL-KZ?8v*L)E!S*z9|owGq)^SAmUtQx`DApK6%?pUd$4Pc{?(iFe>%j=zg$+w*+ZSJYKup34d|Z${e!Gs=`UTfDGfM(vgGiC4)#7U?QqUX$52S#6(}ApR>C^KwB~kszU;(*+}^?szj);nuem>n!bI!M8H6*z7y!2!nmzDWwnr;Nip#2Q=kn$0LC{jlQ<h<K*^uM(Aq;bVt65qOX;g{47^FlmM~Cb}1Y|`fNOZ%!yDU42NEn6<TwD&SC98Z18lju@;}4;l!S#aYFwQmfrxzjNB)geN$*em^Hp(wu1*b)a^3TcI`k3X2<~4$gO<G)s;RXHYe$sn=6+lGG$W7r?aC*I&s&r8qH4{2}89_ikDjS$jkqUhG~BOPmx%&6Srg^6|UL7lpIP*9<&)Q%IHSnb!&}aVF8LZVKUv+=mgj`E#EdpTy9d^u%W?dJtQj-qB)zB`Ke`<sXnZIj-RT|(<1D?sX%{&z%Z!piQxHN2HsNVyVg2v>PCNc9eObNZbR)@^ElIvuJa~XN3)@XY4m2ucL65`UroCMD-Ht{hp{v>V7B94ll*bX*mqnc)O-}TVY%g}^r|DGT!+K|0;4k<yyrOCYh>z@g=HF|qu;5@?sz*M)WKwcQnRQJ)@NepYwUrfNwn?^xZry>GqSK(>q##zGHG?zhX~!RnpIYC<K3=wHb=13Glk4y6jI~Hg;>}#Z7gwbMsigS{o=P<$mq_zOK`fTLRSz@QVpbJ>)qy!m+OY0sWRX*vdGVk8=ZjaJ9UH(tbv^_L-T%4OanVKNhw0l=n5*@rWXeYd$EN5q^rUB2UKVJs7d3_89Fp5Z6Ez^npRRvLJ>lQp<|>jOC=^26viBm&b+9TXqijdolP^h>a=oWp?RsEMNcMBA8~KhdYP$f;A7Gf{n9ouRRmPqTC4tk*Z80LF?0v(ave}LsZi|uua>Q{Qp}FNmV-Q<W$1#|`ufP}7~lD(R(BOcY^k5kqmNKtiV>nnW#;J)7ePvgwnri*D?Y#qgVj-lAmn%;``Q=j3KKMWcR+7o^LjF3$ICY(4(>!o%#;R7XYQlRfD*)d%om$i>i5Evb3bofFp;>gz3B=#4)!9(zo&8wa>MA~Z`9Cujg?YN@nVir<#o&D(jk{BaGnH^$>uUuE>TkE3`J8Q8I+#Rlrj%5*D;wvUXZ!A8`|0RIOG<1I$x&C1?j)2{_(NOCIG97zI@>%FqX*`grbI!_sjrgOJ_|*sf(U_BQQCOFw8l&FQSrT8v|G!c*Vhr&uVf;Wt2H!HNmGdZ2}TXDvm2$<_lMTAqaY^b&58Ayk8T(b#;pHmg>KXkODdcybLSw)O;uE-eW8!WTR94HTuFeI%ncz;0O5nluXhc%}ib_jAC??T+->i%kA_j`+IauG{vy!zXEH2s-EQ}wUN{{jN|B6JxPsUk~m&=*$Ks)vB9WF(>)4+pJ~g4qxvkJBNWQ@gf;-WT{X;~N83UrRhzrsGkDN^j|rqQSjt)SGJR?9^r}pI{EA7+4kDrpM<TApJ8znzkBK3HMxbL9?WpXCcTaQ)q2pbHO|ZqO?`6thVs{y~xFD2ew8%b0hBSt3YYb8(uPc}q`o2)Lp$q`GL?aad6j>H>{R=h-Nr4p#)dY#dP42?br_Lf&0PXr~tr1GM%N=439UEJr&kwX40OFVzZ4-+<-KlRY$B|F-T=*tAFoSAZG*NHRZJLtWJ1&zSvaCpKsic}qEh~ixp-N6@5bOrMZix)yI`>heAl%QF%P#Z;=7b@`ar6M#0!KX9I<8&RWknlbmudU~QIV44u7?s2CM)|LdQ20F0<(oV0VuS#R7*pmY|sk8k6t0H4enG`-Q$UNm0r@G)l`wW8bO+9p7s=uqGE~&{V3-7Q`u08q!De#7CagJ?ZkpIiOOup=>AxiA3ofxG<~w2V{WX=4E2nZ1cpVJIj1P@vuCvN{$6ESUAeg}E>IEbMiPZH7b@md4jRflfr8d=zs<6I*?mCSk<{9X<KKRx#h1d|>^sbuBZ(FiErSEp83(<dVe!OG$WC8#D;Bhry%*|3=f;Z1XrTvD^b-aL(>XhN){-!oA<@yK2p5B+Id-Mj&p}{RnUs`5lv7#3KV+t)?Tb0Uaw-H!m9WTHm4>F!K4odJj*jghwWS>=i(S{9i^w}vG^i8!_lxjEL}>zrfL-{Y4=Y}&8h!xeh3;OK%enAT-!*-GnWfRh&dR#$I*F=2BNGS0UBtS%?hC@v7UKhp93I`H8P&tU-I>}~uQy9bPXkUD7>u<dEUm*cRx1Y;L=&N{PUrJ3yHc{AngT*IUxOV6FB+D+EH50lPi6qdb_l`ZIh1rz9TS3@BAMvG(@}@ppp)XP?($Q7tA<N6AyhlvJqQY`&7gxe;E5Pby!|jBdgf;@y6NaypfB?5<pqMZ$RUrSX+gL<Zr{6tRy0Ci%oZ2TfEVzR&OO|UO8KIAtrB2Rb>P)!l!G9o*YgF3iFDHzxZT8K0qbL%tR;353^cG(Xa#HDI#n^DX*L&uI<PV+ux!GbAedN&{=zYqx?MGC#Kn->zJP-R>lT_5OAY(3VTsAj8})Pm8_!a*+OC18QWck`UOA6&6sLyh;eGJyNqhYU1skTfT0y^5;5aA)FD95FdND02I_3sbOQARuho+D^=$EyDL}6CcM>rhXR#eX8C7xUr<SIDhOznSyv-()4yOHRQWs6>uM$HhK3A%Z#e~b_##zRa14?{5vh$LhxpfLe%1%Tj{LB~+o@KaVZD<gRhG6W>t!9w9h$XJay3e<E{rKwm3UCS(b>T;wP$szRAk=m9FRg&cNXmyY>fG9;)W1LkD%Z68_)RTvM+0Jy_VNuuV>0*JN?I0rIfP;kbmV)PA`zLWy=ZPWX@fR2o(-_L3GYJkHL^VXUEL|AxQ2uv}e`ULxl<(A`#{Zxtf<1Q#pB6^TXgcl^M5}?Ez7|l75wCRGjT_?-9O}LMj5AcD2l(mk+zJsxTPTPbgN7ePj{d65)3zHlJ?!*c;Js!a)GSmHvuwDO^>uxTk$9p`v0bgNY4y)?{&9|;q6Kq;Ex(#==V{`G-O?a29&(C_4k~9}OCG=u)kfUXa6mx1je~(}B)Ye8=-R}LpS;&}H*emj8B%!dH@mQd0W=G%)gOT3vVc2<{^Ey(&{e{!A)H@>Wt{d<U_x#cUMDb*vZ^)|aau%~RR`p-B7t&4*W#*FuX<+#qgS;`&1E*fK$Al_Ona)%M$Df^tRJzWvrWFp$#<T9UfW1RI+Re?+o|ACrQ#0xH)@R-L~aSgV(Ro_Y%X%)q2Xh0N~-;fxumV*yl(L{m-Nv`9<ujSJ)U&%ljq>72S8W&Kh-^JZyQIF-~B5F{Boo-q(n-xH%b`5=QXkm__W5_*)IZbK#3!XYrd9`wG>1D{kk6AUDb~f896yD97MyJepXj^*So^lOLF$<s7sxE<f2eKEQ1(z(n4yr)OhLv;)=syxnhC~HA2h}z_`qbg^8nDE~eL}ni!%&hImG~iD{zb1{Bgr_?3ZMXSfwoq#ewK+SqW&LAcH_`%vKqSBA^A{45U%IR-ox<dB+OPxVn@$cQOya1xHn+YvWfO6!IfrdXqMv&A0wQYWas)H0`Xu?Cos@Co;(_u@p+N^fUCVuNVez)#ldEWXPv!do4`3)W@rz*?b!nvHa7Ytj7E;0QWmZXCpF+CC0Me4Fa(fvV}`#CyQ{DHRM4HHdv4cbtsvAR=C%noKz8LKYsYuX*{h7;v~R$ew)C$u40!-h`c|yM@@aL*Gi;zC$l_0@wi(-n87Zo`2O1KcaKvWpd`=UIn2`^w6A_7vh%&WYo+^Z$RyC=ib&XH-n>Ty>%3;^FaNsW)<V~w^nH(3`o05O(rx)l1p>5c!u64I!8pa(JLTY?T@&ka1>PQ5=#Ek$}@0yyNA5&=Dywm14XI;^dD5}H2CuvdK@{8<nSq7az~Y~SY5z@uK4Og!Y+uFISMMmEKoP$F@V{MmvQTg)39GJLr9E9x{JkfVFm<#XWRc6?Q&}H7AsKJ1O7hSl{?OULN@@L9$cUR;GaPO*D!g8N*qz{pAripfoZ%>%0tF!qAIUp<R1GJ0|f0ANG)YH(n~qCjw8Fk_*l1)Y1-Up@|X0t3k+fH?Hf1#V{A7lRRf}ALferLhIthn?`hKCZDz|ahO@8`Rl?D5NIzvKjOp`I=LaHz-vUniEXnGxp?EXs7pbq{T^UMgQ^P{RN$Nx7+U^#ckSCTHi6UWTo_25nY$~B?4?z>T;WPGzmxyR$A3$od6sNnXR^V4;=^t;M6|XK?f8uv*aAoP;k<&|K!bhsL;T^;(2zFHkZzc*z%cTsnIygRVkSAjmG00?_;bfrIvxY`#UMQs{)<#R7Q34(`i4hCKIVyDpn8by(KuvZt-p#fUj+Gm5AM^Z+jYLS>k~JrwRd?dg8vC27c+YxP*&P%C>pVh+2m}3;L4=S3Pl)$0IM@-oY6!qgt;7%&<Z?*uc9J0rv^~(Jq#LvXPTdwV>rK=e-P-E@)@5u4LiPnF8ekL+EroL;osDX<W3?Lkt;RgFCsK7i_DS2l4rtG{Za<5OjypW!*eM)s8T6y`-FAn0WI?B9PNb`u6vhSf&0?cHIu%*B_k<|9Q;4qCnm|NJkh2ZTe_5_KTicP4k!3j>O7dx#yWwm+znY8-3_%cK8AOyc<l?3tI?lpTs+Z^e<55N%=YzXT`rn`K3X^q>1Y+q3a-wC8L%P#6hPa&bGk@7g+D^Bazmu})<|ucDG!RGg<<*q()DZG@ljtTOkVCbXFP-sE7!e03@7&!U6OUyWpUMHi@uISx_#V<Fus06CR(;r6r2;KFIVWfnG+nGwJtH;QWJwXU)p-m|klYts8Wj{n8{nm)V>m~aVPvnzVK}T~0csQkpUVn%>(3ILCghK&-~A=|4*cn&a?!iIuiRecWWSHW<jsm`iG-FsguWz6A({^(CqtNscy}6lLFh|WM?}DjD_P^N@N9|4L9N*wf*sBw7*(JX$8~u?Ej|o`4ja<%m0+&canx5-vZ<GFguhQWUW=BF()XMh1aO)xZ+vs`_mJ7|SDDoS3^31aLH6wqE^WkIyvR!@)n0khe2f7iu$P^Ma>q=whmzh1T-IN)^rGXgL@id;OPh-xanRZdqLtcn9%cz8l)fI~eXEaBLnCZ<SCA@6I|qi>J}Gzz-lN#*jTb7e>wwoZQo5mfyVV2)&0204;aW>ipu+G`i>)_@$j_{d8&kt~-fa9kQC_bnggmtqOcZu(O}M)L%1^C*FwVySboDH88K38%3Iy1A!|6y{;ch6tj5lm}%GPOK&&*pW9dZGOT2ElF2!t!c-kD_amblF{i!b*_5iQ#AKSi?O88M&w5JI|&h{ffjz?I6+LJHWivGM)hjg9Z~jZx%zpCuYJM0T{0BHI5R-z%K~+eedC^ssP+IFKCGKeFwE9Fm@X%DRLl7CAn7RJbfPhz@HN)WDb*$(eH}4wMjm_B~Oe7=9KCWpi1KBw&G8rcNq3Av~evi)8D>6-PBahmeEE^G=Nrbo$uPSlW1Kh*+8auy8)Dc!R*9A(5qwdbidoN!Qk#L?QN)wubUuXDF{-q<N9XlLz*D%HDEl#oj5CM!*i5+0&@}{}yHA@J(U-8+`M2W)R*hHpvS)8SAcDFARJjCPw7u)qZKHPYfjb`3pQf6=bz$-x%-Cz#ccK<pLE4##9BpM161Qr{cz-2>TXMPyruWl`0M!oX^6+X)k{$a>+O$A`?K5Y!^p5C?rI6>`|^uupYT4B}$DNLp~(@I%uW_rOk2P`k`p`{<)kk+IBP@73+-#U_HKJ7(m24!;vGiWY&f9S-u?BNARw`{)`T!pw$2~<?6a#Zz<*oXv&w{^Wp<~<fsARap+O#Er^C7aMc3cu-3cPmNiW%PdNgos=0ad_QNk$l6m#8t-EIVx=2bp1OlzWu=r>I@^_Y7y+y&0KXo91<?yWc+V6|^!&Cp`VmlN;>A-JpKAu(A57)DLGaUFYi^_B*D{@F{Q1EbRVY~>T{@~!hCs{arm!DS{cDuS=?S>o;&0D}&`|qLmN5m?*<eJ0wF~n;;?)7C9pvE+t!(->As1rVpRhiS{gxv0_#<T`rLQvpN^li3Wt}2S=Ic&E+v`R6RD9fdpv5#CtdVp;O!GVS+a(U{>@N@wGq|6rco&lN~tU>oNcn0DLL(aE+{DO2L{M2$Sfn;X4py+>yr=adCeJ30b;cf~01Tht2h>Q&#i{%KX2F|EgHsmUmVmvy&LaJ+Dwk&8DAs&gzoBddi=QtE?xadYr6Nv2=1#q&B$o}y;*9<mL*?6E<NFfZ7#b5=qq>UUznM-ZMb)^f+Q3oZhF32#SXd=-b_wX;p1>McB>UAK^o7A(d9f%>>$A%2Q3T(BIXbxi0VzoeCr=bAq6#2?>5?J$+ni+A>X&yI3grg4~BK1K`;lr)Y_*TQn^9B%=A$7*H_P8`7K5rUI#4%Ki*0<x`ZEY9cZ)u4=&f}%B_?W9CvBlNG!c`PD7pBb+7X#_a9xStk@%;!R!Svg2=OdVAJB^oUKLp0R7Dboq>8&|HXRaa9EcuMWon(bNpzx<_Y>0Oi;hk<HD0$iKkBa{t7X5J0y)SD}sZsN;xT;Nd1^={3&F~f>71R1b7@Qn>Le%%y^=hl=oWd{<=JmM>(;JVVHJRR`1<G`Tro1gQlrzvab?vGwKn*E~w9JfCKubh~CxitS%VvUbQE$~IdLhVaz^r5qfpnq*WyvIq1jp3T;n-k29mSLgTVMp-uL*AvS7EptOo_zeoZ2g++;(bg7sa|aw>`XhbSyo_vWhLJ?b3>{deO260VDw1fU=oEP%bl|>)CEnTRKsNS=b|u1G)U)jn~XKSP_z;=P-Wicu~CRA0HQnz5?GIbk(^sM9kfMy#DONyg=88pMcLjH0Rvk!J2#v4Kw@mn@{gP_6!j>UUPMPjoa&C4lx8DDsMk12x4EjA_|)gKikY|^k9OWg1C|%7ZFT~?_G>KGT8LYLzI7nWP*m`<yf^COZN!KEdx=+5}g%17;YIDg$fD&y#3^mOEyViZyoXFF?7dZ{kTDIJoWL=qF<bHJptCkfB{X~4ko~alcC*m!ERj!E+Qs!01`B8#kx+E4^eo|>TbUuL_Ne(>FL30pj)U_*>FZc<RPNzL1_}rwO`0_1im)YHSoVIRVRMrZaFvq)~h>n7V`-g_}Nf3khiZ4HCowh<=1snaUQK%5*Zq8&l(WpI4<f*6$0i!Sthi3c4Ko-f9ivwhIsL25WM&_-~+CGLJX1l--)G$5*XtV1@4kD99msmp-=tq-VDC`G=PkypEt$HcgNqIv`V`k^k6+M<|uf9ZEYXF6~4Nrtwj$k&umkeV-P|}8bn*q11vr5{mH=5Rs^tosVFMl#GL4><(Jw3A0XWuM2eqRz1OO)n@*~M*S!<nvn`#*dxMPzH+OS1Pp8Lb)ISVThOnr+Gs7fK4VYHLW42?Zj=IP*8ur;7J`6(+ARq5=TPND%gP<&zOV5@z$2|tC$K2cgCnr*U?u5C?;x7gT%n>$T6k()rWddxcCr9$s6lK|Xnty=k<v0|*^!F6K<Yg=WDI%2LNf^=}oSdG$lEM)7HnwRu|C@fc32aCwVjNL+j-~yjSNuJ_CC-AIQ{{W0?3gU)(_iaqHU$YmpSgkA@{8jOZQ(J*uQD8qSzEJAzDVUSW>~5Lp6>nMMg?Ho*geI;?$RK#SHBo2?v%5TCIWr}rUrRR4Fw8ID>6TF-3-CQcSQyc;-k2tp9uj7qXXmeHw&gju+emAD7Lq401P$WAB#cVJxzgwi^Y7IsGfny_!_KL9a$X##eS!jxR@FIbD@IGT8LnZYPu$l-X6gjGrjKaK_Ui&SBGp2tDtlm9uP-=L4j~XP#|{s@85D~)T_<Z+$Y_;@r)9EaYja~_Y3Ck2jRSZkWPRjBp9XPlV<f=E2?x88P>piS4^fcEbQlJ0^!EYvpX?dBC^>A8a`^~dU^jqyW>c#2@10UXy~C{w!#f;$zX5}GUv&Q*x*jr-)!rZgrKT^+SGUE&@qFCuq7zcy2qf~bRqG3^x4Q>mUJ;BFFY4%7roU=AAP~D6@Ma`lr=PN+kQJQ0>^&CqAh9;kp-Cx&WjVuyKr9gDVYMl>GK<#ke*({1-@QlG=-B#mc*M&HQ^*rI;DkKc;FvS6ca+<u;yid1m6=QM4587+F)emE*IKBJ9a|o!MHlvmX1f(&44q7Xj!%k+v5EmcrmP9YK|G&ljhvkR<@01M$Sph#kg44w^PGSg$INMd~jJ{a1x9ZCMMb0V;YY+(i>Nt!_^zCA!RE46p1_IX!s0Hk252-Nm_si5X%oK8aqBGa<+)7D4P$`v=}6v&nt6>)YTPaO~C+##}zhj)0@=)t{2~=ueTL81ySM0>uOeyKab4`WV^vW3AAbXx`npv=TgD;(t%|(>Zs8}oE*{W3<SBiJ$SuOvkn3y8z$p0EPd1Oc0ra?cP)@{wz!YpcA@oxp;5RoXq*(l!8`lR1|a3R{UlOgi7;boUF{WkE?P6^5wD7!F*FAz9JcAl9P<n|BA5zW+0o<P!C=CM>~3FjC;i^>>*8g>zm9v~pZH(q+ra)BaX5nGgAb3#7v7c)Ome~VZOFf?U+eXf7?a9#Ds3?Zdnr&`SRA4b+>%T7h-11g@WXBqeG-8zD25uP?|EaB>je2Mp4#p<&r(Lgm)aX@;;sRRfWUzdx^l%51v){b2eW@$)nag5wQSCfI8=j#lidpmbs{&_Z)z$6Zlb+IZw50%)B?Tf-*?H9z1Zxowro3HOnn|%<hVj{GuWTV8OB!0;IgSofN5h~3RWqcp7891nD;}}HJ49X7_ud>z-MZYJRH?vbPo+if_7|SdruELFcr)QWSsrpu-H5-ws$q?j5x!Ov-5qn7@hL$q3N3W`}4bhd^oaFr+U)Glo+;iG<L&43}w<;;Iak63TtCMO1YDuJ-aLJO|RMO(AnqjQQ2{$XaVYpXanFsrN@^(kng7E3|U{_0dH!bky7@fdoGY=vxRGxa|Uh-4lvL0Pz}zQ?#xrfmbsB$wlV|VfZZhchzt-|qJlG9m4ND>Wje_5uI3WJ6h}kL`ny-pM??|=8k_^Ip5;(gf-X9F>=<$~4Oil__2q5~N{vZHp~8XtapU!HIIFwgXPU_DOxA&N9C~ZP7-~peT{54n=c}9PtnPg>4@(2)hd)o(6kL<D>kV6rs|pNfIsK?ry+R(q@dg_>4~NOac*2ekjh>*sxU15u)38EpJ8L^CoT(qI3}|x=ol(bf6fw}t1||Q9kMg8%ldGTA`0rQ!3Elgs$pOwal2@(U`dI*!8ILQ6R%|`HNA4+$96{14j%o_-&y>XJ-Nny;y?9ss<K5ps{`GGc<sT9(m4i~AyWo>HQexYRltpQ{Pz;k?IaTyQhTu8V>L|eDc@gHRYnc;qwSmLI+-eJq+boO<z!Hg!4W#+UwP_aC!tg-Lwb|8kb1Ax|!1Ojhm?Mid6C3nRKx2S<Go4V5_vzvWOi3rLL=GusLHxO1L%1)~1B#)^5TY&fa=dp*fmsW1vGoar=?ytxuvR6==Zk_okVS2RTo`5;=C-GlgEmXC&b4v7+68Ggp0EJ<AL2aFhMXowG4nAqgn=MvBF4pPI$Lfzqb;ZY8ar4TT<VF03_!Jkjf|RMhr$N=nehuTBb41X$MCVhE+A1GT|71vNIIs^o|>~)z2jGdGpRg-vAypo4H~?<kk04dN{^N@VICyaYzG)BpPm6z`sxrh+v(ZUYPSCOr>_T36r+P5$m!$bGa)ry?XG51Ly}LL+e{YAwF%Tf%{q!<>y25G#d5M-FL&Kf2D8>(Sl%uzZx@60w)RvLp6#h7bV@)#`%^;di!DW3hxA{aKsWfxV_KtX5SKd83<^IbgAF9tgd}`arDu2!*#1qP6gMq2<?<Ve+uIaxfBEy9qqn;c=9si2G8_G?Z5wk-p8MaXBU%!hb@*FrE$w&Q4s8Xho?&~g`4t~B6#iI8fpaGd1%vbX5yXk3+`Ody8W_7#;GlZl){+|>k#18Tb)AS7gaoeety(W`W73VM71Ji;$pq}i@FIky1&j=Y7B;4RS={)8@%*cw-Xo;xfL=RggFAT&&KX#T@U|ps0$~1*R?Ecqxf1x)hV)ysWxF=qoz;_ht;R2xy<V?0mxS49=jOjr4>An3HO9D*fl9sI<{!MLq~aUyqs;3p-eHLl>qyr-v;%jjSnY*@$?AupX3&wiZ3CQieLVp*D)(Dp)__AEb_ZY(HJ`;F-=@VLM`k9@hUyz02^NB^0x;WhZ5W~D#G0pV7>IEVWp$Y^CJj+k(Xt?ml(-7Zh!^z^S!-tmIi9ZLeln}1;dj7Fxr+JK(rtHwD5ZbYigABPLh-Vp?imkSH<$q3<M&TOQqSuENV<xsm(GJzcfj~?uxMkttneO53N_3}%+1AaS}3sqH!WoQdyCbttpgXqEWcr(DpRw~ZI|pCa5JrElc9y>c3ENahPJx{1inLT7zdR`fXXW9`v7*N)Gz3Dw(XAj<3q0)CW4AEHTSY0h5+JApRn$IzC@#$uilT-j+wr($F@jzo+H#&#IM=U_bZ5YP_bCL;?|&$fFHfJe*9ls@VVLnt$@?x^h@xz=ijzFU@4A8Ip$zkR=hTaAG0nYk?ezb$F8G90_lh2&!rauapi_(+jhT<*Yoaf<+B1#=d0NR3a@6<tMzm@R|&j1d4~r%y4v~z?|Dv=g=<}G`K{vOTDaC=GBwq53Psd$&48zk3VVTtevvI%jb}4o8uxE7H0}&}L#kxQ%2=t#WHx2CYflgGbR;_-7IQO^KQpCx7XA_r=Rj6_=oOFMrDt=(GZMqXw@0}#gjbDeD}2>Znn$fKqcV=YS1lK{I)Y^^JNZ6USNEk)YL(~KiHZH82CKA8G%gC`PSwzKC)`oGwzxZ8k~V*vmNNe@95I-77ao3#9ML)~nR^wkA-XiZ=Mv`nilsAU@v_WHwD$05Lc=-~J3`_|6k72B{%gawfHpvJLA*cm(zo@Jj0uIydk+(HN(ZAPw~lsPPEJ$q%el39SX~$~M84|SJmS=8#ogJwJPAJf`7!}F2$ZD|@Vw;I2?#(yXva12;sV>#?<59@L!Kkv_uNrKNp@K6@vW`wI=_VvT~+Kj=gV6j$pw$K+-T>;A>5c_VP$mfCMIlEjc+IrQhp?o9e}LC82&=kz@z(iz1*!x;G-D`Z=`#marwSeIKV1*-zFh!>A!%CHr!MtOi(L_RGI^b#Dh;7W{P|Z8RJ<TLo-L=%y)HE&lW6_f#McUW^L*Wlqfi=fP^>x7%o9v!fB2zNFn^reul;V!ozOjwmmdu>u5tPlj>R5O)n+0IQ-Ml@2oDEhP_s{n*f#KM**};0jN=sfHf>B;ik(9{MJ<>XzF@7J8juS4DL}f0CA&TvurR}gIsHQE9Ycm(fIaf3gq>cRg60IV(19#sCdA^G$l`#zS_N7W+OXSDf;-EqgB8&Th{eSH8b`3;x?dwh+Nrrd<1WC{xHFmOpMUdX8;$aGvY<)H{j21-VOjderzm_`v?o;Q<*qYoc{>k1j&QN6*2D`P8V0qzk1@NN%<xCyegh{R$nc?;&=PwS(NH{y)pBcP21!3?Z3~9zo8Yycy{^G5JGU;1KEjg>zEbxoR2M239eaF2cLUy*0(#D$_M<?p46M`^%R>LR@G#AT~!@XvuEhVYRpyIt*+fTAi=*|n(G7fPQq*i;>5Mwe!_H?mhWzLyg{2N(%2hBkTZ>7+8PWg)l}FpGG;UT{>p+3!-kN;&rqRTL!&o$_ulq3U2Oee`>z@+U2K!J2FL0&;>xs$5X7vFHt(>M7{>)3tODETn4$c>wEp0NyWi65jPB4G1=SVk<4AP)+T%q9Qxv_Jf;^AyOqy&>WGjgPeozLEuqRyn!!+Ad@7(dU1AUqHrH1pl$xS;-3PXOrXzlWZ-*TCB(G_05c<3~{0tan~IE8!KbC!8CLT~g3xv%#yTMSn%u;26Ed>NCzFb*NcB8o81p9KkYYYZqNqt=-1a{eMrLkL~ru~2zMWRTi0<~qF&+K$q(n$aQiLt%)!l-)_I6VD}Wp7VZ)!+c+YDF;4YY?Y8f%yP8`+5%3+-efmlZQ4PcP3o#ccIFFn2$h&112g%OjYdJ>F!v?7MtAf{08L4)#d><Zt+0LQukFf|JgfXU;#bwN{x;lVV7@g9g#aHP3?VpwcR$^>2XOkC<}36FsVZ!`s^BcFD#jN)7&|Zi5A*|xxB', 'NB_FLUX_T1PM_engine_repaired.py': 'c-rl~YjfL3k|_FJ5&jQYcI*yd0~AR~ZZ~aCjI43Dy+?L?WxIQBa5Xp}35t+NfCfNGY>oc=$wxi$pyZzEjeFys*=-91s<K{L`KrvU@BVFnQ{?-TB;AkGYiGSJ7g>7nWUtfdJlXrSj>;rjI-kC{2cGk}|M$CR!M!uxOp<AE5*5iR$(>1_%r0W5jEl1SWbet|=S5OD@c)rhB<aO64(7=+_MK%mjh6c-*)p0q>rEa97tsc){hBO`EDe@XStirixsH~}4BE`nz4&$==gBHg%dX?RD-mRpmhclkM$6^anPzDb=hsmBIvzU3D!YoED4jX$WSN!DI?u3S=tmy!&64OM%?fCu@SU3lymyvS4$VWU*=C9$b%i|sx=C_`<v<^n$uudQY+VLPN*&pQcFQP93#VM<F#yUgk|}klh}Tgbl?Zzq=fw~v!*Pzg&YRn4S~~M2P0Cn}w9M8SRNXrJVm^!QY896`j4RH|WDYQ4O1lUG{`YwiJHN$w7OXOyk<CgkMwI7Coa6MAP>A{z%y8ZcnkZ)x<*O`BrUKG>8GYTvCG-TI(%3215%lJ5IDo+&!;)R3@eF$uJCkga&LZd#jJ1rTnN#LbTFkS21(R9`aL3S%0J?z_U&b(f(`*fMSZ35+Co1``>^5PD<Gk)dTY1d0pG;sm7{lJf)W*(9lrJ-aSghk|nP&)e3+uX$mq{9%bMijQud-`rx_||7HU+diUBuI?IG<!uK0~MjFDN#Xi#*$`o%b-mIPEZ3oAnIm*I6aE@k{``Ny-JhnC5`0DNY&8UUnVlK^`xoTf$04#Z?>?fWnC5tIe`Z3MdKl1skGpvblg(h`n<XWV0Dgdm7~e-2rE^b>Z_|i2wZ@CT{6DBd6G`!dvGZtZ?pp3GXhi?%i9z^nAy06aUh`@|@{*3RtPD142(1_UcIVfd0xoeEjhZY?3(2VORRl^Pk@Q^3iuDurIIF3{0}DECA5jiSs<mVYA>K%w~X|)2vtk4wnViEi;%B7zdOEXhF%y=FmL&xN-s>U}V06EnU#$0jh6c4?qt}M0>FzuCwK46+0KS9I#md-KNXUES?EmIYC@mZzcd`5zhixwg^!N_88UaYGgQu=F)7I!CD8w=NP&!$xFmr9H`4iG?K?K76%r$*uaaw4-oTLkph+oL@Ib%r+Km_2!!VN6=)xjhIs;;Ws^>`98si%A9;KMh=H^5l3^u8xzZS#0=h7XmrE!WAuXFh58<D*w9r1jPT<`X#)YIb1`>qC12z><mP}FGq>!u6hysMlP5y@H6MW9yzy1Bd+kfsnbLjcV+wTv0@at#z@x6x?0ZnjVQ{1qH^S3Td9}H=;<b{f0)^L)uyeuRg%%}rfCre{Sh6rb%gq7OMv6e$X0Zty>Xe#3r*(Qh42nwD8Wscxq!iwKwxx)cHVb+q_CU&xX4;%ctiIzc`AY$Z5hk~L+$~w<^DZ9?kaz=2vO2jAtPm}dh0o}vhy98{`;}uctNYQBDnZqDq0VAY7(AR>%!rc7VCYfG2%&7L6Z8;|&zc4-w_h{1pJ>R6xhu7hcr(gaSe(ryKAI9kgkfspkECSv!>jJ$B0>M(4H|_0xlt_%Mt4z`jka2;@S-RZ*KG+NZfD_k4{M5h=mWJa8oTCL|c(h*8W|oxuP_E3%?V2dVWK-^8sTmMkm_F#OX3EY@mS4@6*^M*HVqVQWo@T(fpxtEzJ7EDZu#pww_<FG|faUJNh>EUrDmkvrW~oP#&jtZ-((EQ6?i}DuMANG@yMaw}5tA$+wgqoVB0%ecrjGzYk76KVSHP?<0SBP-|2vpFzz@=R*}Vi(@V_0LM4G2O#v;WYwCH3=sbD8#ThL~HvBAcQ?jE468tM$bEi+(*Vq6Z<BACS$U?_afVZLCT&WFwF<97ekH2JtCT2pN1z|Io@$U6_QDZY)T8^+|H%%9A}BHV&G@Z4>H%r_#Uak2rZCtv;woD-UZt<j~Km}awB&=*k_CLU%jx&T144H9U;d)Z~E3iR~-U-!>`K3TyUUxO^dG7^rk*sO8SLt~RI(>ZJuI2-VYbk=2|5C)NFD<=%+n-aKS7&<@;p%(NSU=W@Vb&_xM3)pf+EPpSeVzEpn@+Yu>yv(v?Az$DssmiOUT&N#ep&pB^DvvWEx`gX7T}A~?p=`YX=2iOo6Kwn>Ud}`fMtWKJuT=CM^a!bf@0=zD{C*#;d1w5*X4%tsJ^}6`k@lU>oAoj_Mw@O{uss3y)3s=V-gNDTKK%7K&!ep<QA`u4BR+wI6D^@JkT^lS{tcQy%2({|?Y;T%)4LCE!e8E;eR}uv2O#yG53d8H5{{${&Y(AV9`s%WgF$ES?9Hbyr=P=5CvV@pKgQDOB$zKZw?WxouRx?JKynGLq2S)$ES}S}heb4xUE-47kcbg1_(_%mZk3GHiiN`d0TC%+9_|21fIgzshZ}NQ2LJg9ddOwF0E1v9b0E=#qA+vZ3aks17H#4G@)%Gmb?yL+d*uzyUU@{Yu6*Yj=@?b-Lffmt_3qo@yGvTb5|_!VH#Dv_prN0)LrvFR6pdjC>mVx=*)37tEM3pr(Mf=Zy3IwL`wg9hPhwGagP>I)Or<9i(k&nWy)*$CP23_>+-%*X;5Bt`JttlkvGV~K{+>~k00^WJoT*qugQMrJ=LCO71mI>t>|8*r7t3q{oYwWa7jfx!fNB8s55tZc3(_4b3zLt}S<zj~jx1vrpX*c7*`(2Rlq_+dGIo*h4q{SDHGHR&PdXlM#Cff<pg>*G?Rgg|Ei6!X5#P=dAe*pSVt_Eo(1IKw&z&_&paMXDXE0D)_wEcfd*N3166|5v+Tm5aEk>VnTrMGofQUVEJAMbcJM2I)Vt!#7rK2BV&SJ02;N8xqoChz&oHdU^bi^J65~lB*(ZBM42=I&~L%_5@06PBPUlg@|vPXZMpFm_AvcQ{0D^&N({E0v_AosH&sxC`@GmAkzUm8U~0~Q5fnQ;EOm7k73zxgyIB0om>=kRrW&fjqDQM+(eYrfTzZ?@vsTOcICog+F;ZFPZ>02_6Gl7M}x;xb@Q!W3^2rssKFzl<)BgnvB#^r?fJhO6)fbvr*EzdNPZSchMBemy?>&=IxQIg*fh$8pAY0O|hRxx?n~ht3^0dEfE$xEHV+U~OFxs+T<W>S!2V=fBeqXce7N$9d}Xdp)Z$s%*V6%gLrdGylfc8sG*9cL^JxzX|Yv0NNAJmcr0bFhmg9K%VniP{JQ5pxuBXDkV5a5vo6Ex9|>mExIMPz#QZ=q~l@e7V&Zp*b5qOoQD_djl|$0Mr?*D^rhe!PB-hV!eb^untCD}pvnkBezm_LPLWtb@-NIv$Nml#L$WJip~|=0HM~sM>Lud1?y)wL;9khL_6sV{(i08<iT-r$INzak2Pl<XfW8@9#unWOV$gKmoo;4PSBllZsQyZV@d?WHz!;oAJH27Oy50HddPfO0ZA^Ze%7H9SP`EGL0oQ9T3rZAdJ-T$Q=O1&l7?ep|xB$1C%vu0}t{#eIBbdyj@eS~#Nr#Xb^pj3Ybz2mXz@X-~j0>-2$|~*3$L=g9UdNWFQHK=UY24PdiE;Ce){Wdd3Gm+Q*j3}(X$(5!8~Q_L4zw6d>&R5Y)_La+amj_!UN<D@VTj}tUfg$@@yggyEmNNWoj&80dxH(Rt<%*Bq#hU$hJ#`@OA4eZGpW=o9e`!Bq3i+ZengspB*jao)851+2Jx5GaX=>@4Kihf`W3uj1=<&CGwF~&=g9@CDy+U@_lG;6Kda1XV?hwWQ5}LgwP{11BJ&eFt(!w9vQ_tt{(M5Kkc*obshQ{PJyG<p%tBJV1pT`;(GeEN*p?MM74nsAFDe+!t$~Q5NQz%c4WO^OZ8U^NxoR018cE;78VDB3sV&vvTMr~s5bFTpT+g!sZ__E2t}@97M<s_t^x{(qtZCIvH_K%gK`!zv1zpp%+FHe{EZ>GKA?QX?^tIX~!@)zd*;!;oDL@+Aa-PR=I05}E16&O!#xh%U4eI?Am+#Vf=GIqX&SZ*V6U5OS!`&Y|eR|Lvb_er&=cnYgv(N2}?nHBy@0~wPa^MpnI_GitC?X<*865=$e0pIV`}{$I0xl|Wh1G_QG$9`Lxa*;^W{hDUozoh_gILBSz2-9scRfzdff269m*=D?CrEi<GhW0l3SVoF_2*0dI)QE9_Z{(7ESy-hpwCqD+WB{9z#?nk=q0hQkR~k5rBR6$Op@1y0cFgJ^Yg>=bJ5>yQx4mT2>G7y4tk1Hft^(rsID<Nh09|&@JzZvOt(IT@WMb#gpKcu3UEGZblm{vr0*etUB^*5I;iqV6nzR1X_2Zy=dqqk@L-1jZRGSD*s3u3RwRov5~;d)09Ux~f$tokj)9D9Bqyc9T(Rp@lnwLj274a^K|(Wo%wue5`D_eDeXNAjOWhpf54$9+id25iM3bn%pt%c>1HO8I@6O=RLJ@eaF(O9;XAy^}+>?gigtlai8<Rxa?RF(f2ykQqa)W}`hp?1k&e+Cas3q$7FT|s^_#4|pVUQamxK<}t382gPYIqJ?V?7>T@m9L3Z6%{KS&o+DcoyQJA=(fg=uV|mDBPE_YiQ0>N1Jp26d+XK?^`}^h4phb=`7AgvuyR}iHeq8vi}mIpzzr+OLDXgpKB{9DFWd89G5?D;Eul2LqY!C>NA~i^H~p{w$r&Xona8q2pB#cKj6T~rqs12-<{*e<Mv)bA6a__^*!ht^G)QY9npN}Z0OWz%MNq^G_DYyG6PKwmz==sL0_Bu^+0Y_fM@K|>v1$GT!H{?I(^)AL`oSbC@MUxps(xPK$$b=FV3}o72PIC$^?mt@{3h;OHTm&M$ziZ)^(}YGn_-FPB&wGb6z3sjQhB$^`rt^T$M5;v4$qMaOqiCP!geuUeaGp=qqz7CzVM8P}4}%o9KGY%TVj!nAp=f3?+hx@umDV!CwJ0FHe0A+)MRhqQoSWxNs1NzMYSqdQ)TSBoX}chAL;fstkxn!1JK3`lr_F1FSBpu%hw5;w>u~RK00w%6>S_K#fjg6sL`~QE&Ae<4MU*N<+>ih+?iB3ihdoe|tC_x%Xkkk)=}Uv1;htVCL;jLB+d!gP)@j8Fp1WuKCoS$;N>2`AqdjErizI3S)SU?c%OLRhjDdJk)Ohh_wywu7<Fq%`*$EBu*@F9k1{8A}Hq#X#O#a;OiRyKUxnpW{ZFP!AnC<E!+n@ta4{Vc_P?63qIHQhP`6+iam**R(dxx`=*MReT45JdX(I{w>5rss=H#y^)oyY>gm;ru%TO1v!fFD?^EX#W|6e)$OC!()TZ}P#OTQ)nZuejbYry<m>bb?x%hGV5Z&HFC+Gl8>fhV(YJ=<S^20L$k2==ChqhkZnZgOGT_=Fz%lP)-uWY2@?(DHI6uZBLg9d(&Ldh~ouL|c?pq*U`yW&AhMO_|iqpw{d&nd`LueC*^;;nNaj;LTyT!pJ_7Ov4-(p^Pmp4>J7KofAXifv?&QsJ4HGUafDP@Qa!o(SS((qYI>1nZl_9815+GIM_9w$({*#Da{pjc~=9>d{57J{Q*wF^eGZ0ar>YMLH!bbkrqMAvhUy3b1z1&{eePj1>%X^*XuE%7S|`2UQ!SQIxFQXpCTnth0EQ!I^4wi{v+AuRYstCv_~l0Wx4hub%NB2*SA602fs^YS;oAjj1<2mmw7uUeOPqil3irNTiCdvm4%tq9Tm~MOfdMM%_FYqx~WuQFgn=X{WMUe#mYbXR9_-q9Fbm)3X@axjAP1Kr?BNcMiCf<pm8Bi!f|~36+__@3Q0c7|80_C$xQ$bjG6^<8R*?o;M7P)(6C7tAB(_Bf>Sii<3N#uJ$U&?^xVZL~gQ4mcXZxeh{|?0umm~TqYjS<3rkCP4oo`bs;HQ-8R68d}<m8ybAvwGrwlOO?<p>$a-sH0?3HnTO_%#jW*~;!KN7~3~ycW+OswMdT*$+zG&L>1Z)4UZqoo(4Jso!8cnim^iHIPsG|3hT*Ok40kEX>^a2HrHt>?XLA%x7->Xpzf;TkPfhK0k9JOlWxy3QkIEJ=?hUb8kFdAOPw81?Y8<U8TNjX<ts@P8S2c<9R+Dc<cdjRM3>C*%8vkyNH0F6_@iqk>$#Q|<g@#D2+(Me(D(?Od}=fo}x(q7tV|IrE+GBjk|wdobCyCOmuTLe37${@hkQSe%ABzOb;#EewTOL|rf=-)Xr;PHOS{QlNoZ4jz^`;1<q{tM6PhTh{dzSOl}k9k1YBn<rWoQOc}yOLPAYkllH_3%+uQe8#15;Rr0)gbx%12{S*YnnZy_Qdr3zCR#*$G_p-?X%u}z|Xg4N#lP3RN|uj`)B;zRun(?sp#N7;OE<}E_*VTJH(iao|+0KNkH7I6R7LmTBU-PQU>(Cg5G<$-mO*Y*~8EWCiMQpeK$&(>-n8?lF@N!sni!HFb+s)K)=aLDY`{}e0G8^oM*TGv#tN^x2}Znj^(h`<`obq3MxP%Kk$M-KSS!>!>3@2PknqRE55Dm_f~UnE7rYjTAm)+m|6<SStdhNu%i^s*1!NINQccO-P6&jZ;0Nb*T%1d(TQ&e$)mT@OH1j6p`s(qoeMrv<>yA3_*Cd2@{pmS&>o+)RbMOF`FOnsZWhVd85!UsA)<f`V#@}DNeq+_?(x%%JEvsl-ytDX75qj4!816lrNp;0Q$wq7LYv+E^s(%ezIcYERN`>BOs-;=XNSMX+nB$@gSL+<+}_#8{FX#ku_S-7j&7SFJ#J*P)y4)4eM9jSR8zZL+FUNJlv}r1WuYO@vqj{ut!2|%*#d`AP$#eoKpU@7Z=fIPX8iTMB7VcZLl#8d;I6y&ov#&$B=d#BRzoJ}wet%4B+Sp1vidM#gndDIA;o_+6k7<73i1uPZBh-;2m&BpRW&qK5$HAyuvRC1s7`9vDKQA43~$wW4mskI0YtkBW97<JQ^X6ZYIUq{fNvC(4f-_wmC|V9;sV5&%qe{;DBPHvKChgj>7$KJ8p<%kPzE4|#=)~&bt{nGc^wR6{M+%#XXn$$<InGoPo0zFv$J<^&YY7sXP@8w`0nKR^Bd`^bAmJ0q9aKb1auvUIr6NN+pebfzV-8ferk%(RVP#QA<?*RApUFESSRZELRD0ejBRz^*6W~ICY13*P4!N?pRJlQ`Y|DIm4@<6bzp9>E*Uk({qq{_F&sWfjU|%a@8SPoI_Tlxz*FXsB7CjTvtkK?DPA+8u8DdPG_&SOTg{W*HK|F___%kj-=C=WC-VIj>~1XnT8$Vabp|3<6BQmVB|e<+2aNy42%b<iIRHcld_e69$_|=}oiIX)B1$YHSKf~Bv&hm}Lfd9GaAv4o_0ZQ%Ak_$Em3ar&O&G#pljYmd9e=-<C{6&iu+!mCV2L~ie`lW2@pah-tb|7C__YQ3<Y$~JP!0`P^A3#RtpmLs-PPdU18yDs{d4WR2^`8y1~9t8z3gJ!0JD69hvrj`PUK~FT%IGqoDC@QrVESL@l|Ey^F_=?6hIk|4*NBOAXgZx$VpQXi3X!8X4Ffa)JvWC#-!mJcQ)`wccM%8Y;1F92kPC)J;naYO?bzD@5Qj%W(fsR4g@trs1pFNh^{d_&AA%&WX!gS6&CeCBs3S9uo3YPEwZzNr~T2LQQllSAzDk18R6F1<Vqt>bn<eMT~M6(2%`^JmH??JqzFv+jELNsUz57f9N_Vc;~23XM=o_|u#9J70f~m3k%?Be>}gf$m?vqDr}aspqhlvqCn*bOofW)K^HJe|B22ACX@e9k0kUFs9ix%ZNkuI@iM~O<c(vaQ*6DQ}hka#r4<)p99%Ft1c8M&K+kiR>Sc*a5DLETJ=T_0WaB^guL5{JLQoF-9sV`x1h_;B)PvZ+pT7avGzhwtamL9R=WwMGcc!q?QhmQr>_Y#ojyrEN6ov{u)?cpMPjqYu<2$S?0kTN1$rFKZJua%)08V8j1BfyHXU|ZU_qiE=j=jidpLy#w0Rc}EQ57GM7fNm%*Cl$vc6ek(*<Ll&WMCVXAMdUvHN$oP1ZK8F)w!9-a&|ehJ(KoL0(U`ZfdkHhKCe?>whj#gf_4$vH2?6&M&7P7F$a)%n`{I1l!oXI@ki4r6IbSC0;-$cJ&Ff73l|i$H&8XhlF?Jy+Xhx_P)~GOT0u!Z%ns}r!a%f8yW7Ca_-gq|S9~=rYC|ocVf&=5%6wr4%tilI|ho5lQ{MosPQ!-V~ST0y@+e87RX@Sn2k%OLY68eTkrlcim%tqM`_c~B&Bl{H_6&$GWlC4VYBSlg6A}NPYuDK!;M1{Y9=bVwB6Hysqo~T<Te36U+xe_Wm>kJP#$ZSS2paPV+@Z<uB?35y^%SDu$_J0I)#%|tcoNPhxLX2(?vnxT6+USuskJ`2VgU=fc#<jyF^p>=WQ`q+&Y9PSgJPYlvoEo$`8QcLjXthvad%Or5bM#`{`CXJODHFV4x)fkYVxA50sxlWQHu0~wTJ}cy(UK)~^8yLP(mU23z`%CrV|eDg9lgc%Lw|wg(t&-YMGb`d5evLj?s}B^L`qDG?Z1yOO8~k($I~kl|32=Y?SG=&DC9d*U@W|dM32oJ8C^>EA;Gcetz1<K(6x(FFO;k=qyl>uowJuRPm4O-(wrUPfP4~P$LUM<W|5ANjw@7J)Fv)B7c5Jt8Zc^kq6M`yT5H<NK3vG5J0CIn^9=v_)G?{9FoBGB_-DcP^y=*Y+~&WnEL|3`DmS?JuJAYuN%0ORCGzB1Mytsza)7;_jn0O)BPH%OugP(Fxmh6-KWpODJ2^2|^_rkc(I170iuVrADsWvkd@%@Q!VN&=(gAIeGoY0Udy03>prF%_pS@v&r}DbmLD-|1Oygmwrq^xtz)}xTc5*~mH?FROQBpSCGPB9K@TP6zj3RB)jD3#JKAs>~zNCF=a!Apd$p?|xFm+(UfZ?we_3@pLqh}nSbTSGQi#8RbKP0f@R2cgW>hwPr6a6MF;?-mspJeGgxe#bKiOXonQAJM-k0yTpP#Ql(ZYK85E{qRpIGtY@zeTJ%$}F|As3KE#tdNvU{4AXy2GWR?f!qk~kTVGP3~esmgPsg+CZlIG#i%@tBQy&gJ?BEm3IR<*#OAGRp)VkyS7EeVFCwd^@o@>&mPPGg8!c-+-#G*oX&_LR(3y6RDo);)vbasH#gxmWSz&=Gk;MGqF@8__*Cjx`S4pZmZoZCgmD`A%ZxNS^lh_+P@4e{t<y_h^ycSz&kKyI-K<s8Ah4eL4R{6EUsQ8+fA!WlFs*ERNnpd0wl?|e?MkoRSA60J!Nc;^&AAqgh7(}|F_`x{2Q;I!88jHU^l=!o%kCN~7FPO!Fy{9`?emR*%PUbQ~QNcMSGW4M|BUg2kube}($yd(v2b+{9ifBH_GN)J}Ii#jU_yeUr!~~3$CRJpf-UO;`{c7AhM;ET%gZ(g|kfg)$p|Ib2m3EndiL)a#>s4CxIFhOI_M6)WNsH6XEHOvnGzazq`bE>U%bn)@d8B53e;-!+ffiN0T6>civtWHF_Xi!Xr8(f+ubjS`a0Tnt$S(EZxKUFClty|1s$M$FZe+R!W(1$U9B`gH!HtX<iAoZHDYJ?S>+p3>1Cd3*6k@neSM4fedTAKIEJ%IFQk73=v=^YCG=%%hv{JY0%2yHLfKoQK{h^dyNBye)RP7R}U1lsP3+vAhbvZsumIW+zO@pYy{bE#^<XJSEMg^>4=0bhn?D{ju1+GF7chb$(oX{+v6$U6+Qq02TY5-POs6=@%sP;xlmM%l=Vw5HZ<cxZr^B+#%>q74^Y8|P4t+L6MRdOIPq1MN0&2Wi>MqVs}<9pp+jp_(M2G41TMh-cY8`=oTmzooy6r&?$zWEF0zDlO6xLjm2jn%X0;v&x7$g6vzMcv{iT1U4qooM?E1~~uer5)K!1`M$!qGr!@+y0ep%Jrrk-?HhB9+l(y)79p@)(sV`o-LOyG(jzn0&?7CcXgj)u+T{9t~Or1dm&{O)t#Ir1?59nQ3!|~{5P}$A<$@6cXs4vZLd2MW<wOJn2x$?#El4a#i@iBe8z1IupX!_0y-6KqBy-!d7IKmkA3HthGlh+ou|$(wYBv$&bz^YR1b`uM{VAi&N71CA9CVI(TwzHG+O!+k}C&q^f3s0Jfa~w73)&J82i?6Q^CrW5McHhVXg46VRq3MV}u=fY&h?iFs*hlt#?o~PX`J>8lE&Kx8P+%h{Ef!K>_}YAf~+8tZMAeV8jiz*TXg{g}nN>!j+9s+%x<ysCEsx#si~k>N2(=$NKvQ8?;np*gJ>#3wmtGRe7d{P($-yto^Z!feaLzL5S|}0&&0il)%!`W!S?16;zq8ur0pAKKN>xA!JY&)HBdMfmZ^J5*vlST<TAk_NS<a{>xu6n{59Qw&d3X^BX!de;ufAhBsi;$9h5A$36oM>Oh&)e5o-}^oG%WN0Qg;@e3Ik@C73{p-+{!mkls3C5(O}jQ$Q7Eu@V(TS%gM_;mxsHcut>F_(tqAVE)zAm4Z@U%$reCtdaxb&(y9Y%$}&_Y|MM6)6BMQ2}A?kD#G0s%cmTWYA;<WYfh@PqBbOMv<^y#qoNUtT<O4Fe(k#?Z?5OXW87!MO;R9Z#QW|L3%-5((s+eclNk;4+teE98HqC)OMZEki=JruRfh8!XSAKd*_uG{pyRsdDBWRqA36Nh5Y*1WjuUn8NaqI;#b@-|I7l4N&Bkj=c}HXuX0Y*<3ZE(Xc0~%8zVtZ2MtsdaHK@LTvcsG8freuaJ>d2_G%ZmC8G8i?ruV)Ji!#C51nmEWu?})4NY2-R~enDQ28&|D{Rtto__UO_*-=VNFg!S!p@o$i)8Edygo7uepuW9rf54!HAhi1QnRIwtn0Vhi?%0qmfL!25l~Cxm8jDDJ!p`wqT)&)1aL|eO)p57_o_1Q4us(Kr86?h&`BpT)QQIP4{F=exkgs?dL1Pu+QR2*<NXg|CR12G0(k(<ECjA`EUyKHL9<h^Z9$%3|I$(|jc;12$sV`YoyALHHLkQfbkW-F>^leD9;T#l5qN+-YXPv#E;fv{@Z>gdETE2kj0EZS9>tVenkl0zRH+GX{=;#p)mPm6o%8@8-z?%hb`gLUIfUThJ9>t`G$N9<IYHn=FvJ*IH>OonZBnR88}4^fA5#T>-4&_OWWPMT_<?K<sy&N1h4QkF5EcZmqr!51=hqei_(6g9CV|&C34ZuhiH>+)rGH%-8}8;3i(eYWFRkL2cJV37w=T2++#G62EmVC*2{&*7D1J}iJ9>5*-v?A3J_r1{QvHBGU()BKS{vmGLoX#6LH>|}0(9^x_VO9_<thI23|oDQ|2)G6;0Ya!<KUpzARa)?eQL+Ys_Jo|9}mQ%xti*DNMy@W?8fJt2P(NnAJ2mx&pKfI%2H*rtFFng>hjkAWg!n!c(M`UGU`O(&5(X&5&^pMFU<uOFmVf5xdjm4#er(IofZfx&_c>EL-l+abAAHhWT~zRDHj0pVnHwYb`MImk1jXEHgG8CEuJoX=@W`>Okca0-n<Y0lIu2kMQfPinvxc=5sAg1i#u{5{t1xjw3O~wO7~mJ4JzdZE#(d><qrN?5u##xXw4PNd%is@>^~zMkWoL+O*V?Vj+7sC7tykTj}~i>LNj*2s?ZHdM%8N{TYDTk$A0_{cK&!#zh;uRd9Uhob;mYTZf4`vGMd73Za0IZhl=Jug1CcJAbVTWN-D(I$dRYSzXtCEuj!znkquFAN0mO)UZ%tly*dws-kL*K9)Rr%6y+H;ICBH{k)NolC7eCTf*xnFZn8&1Jxa~ltSLX*&W!*2h!`0Xac{9KKhlu=u-#bCC`hOuyLhP|Is%a0qd!9XKRyjcLEo~6urLJ(7Nu_UoU;n?fCS4e?>o=CJ+BcB1j)jd|1U!Pe*hu0X4F3A?U-QU08<g-{6bC(ixMOPGjDPg6*`^H&uJW>*^CFFZmHu-=~(Pja?4AHZ$L{5%FLbNMPMo&XS<+`VbH?QN&LSl8*glb%wt5(M~Rwv>`Mb9ZZ*bEoYzeF-#I_x<%@WaP?;y!;-1lgr@Z=nM;)?rEEb7Z!Wo9I{K`yldAun^v<~Klr$xal`LdgU9d+=QfF+%FMiuES!kbFzEW7``Ar6tChKR9QF0JEvx+({a%v90asY;Oi)(Qme*bD?{3%^124NV%mp$`5Ks0V)p)W=AYQju$wB?+fUwyw%IJ5&@#J=2mH-+TiUQ|zpP!VX#VyonRkTu2BEn*1pse~FhopgibVKPVzbHqH&(L--@xm7BLN|4cBkh-y91GeGHg^wb7=v24!<!zLYaRyZL#zBdxL0abgQ1fn2pa`df}p)w*6iQR@pat*pVw!5e5|BNXG=#oY9E&nNxRzdJ-x>zMMbqD1%lMp%Ev3VOOT})hy*C}L3UU`!_P_m%xc$71Vug~PJafUMIi}TChnQuXhFU`X;8N)M+L6==6DfWTx4;ACVn`6i+qo81(R1VLWoko(c3AFEofUgT-=;WZx;(F%3*N$r3S0K))k}vkOfwi=1VJIgRHEJ$6ECHRSZr7O1{8AfWQJ_Kc;6g<sTZC9H+W%C)rVK${YNv{Es-`r)YTqpM{#tLvJz0AdPOK3mlq)?o#B??t5}guvrHMc178ZVvi_(gZ(rCw6^B#fy1b^Pj#YVa6C;zQpH*>ibeIv5SzU*T>?sX6PKJg#er0}PI={o_4g+E}h!kFF%#{0p53Lo%CD8)ZTo%3_w&ORdJkc1~(fAB&V&<nK~?C2l>?!O@2`Fn2cIkp|B--GX|@&17N-#e21elA}J{Q95lVjuG@$p|Zt!#;`Zxr@FK<jwK479La!|3pc52=4*2S1!3C08E|HsIkDnNQ_9CA5wKi3E*S2S(YKZcMo6mkNVFC2Zu)a_n0oBaxCoPe6i!8bd}gqWH%*vf9pFx!GCXWZQJMj22V%dEb!Aq1>QCk(Bb|I<=ogn?|-@ln)Cj)BHX{dH3pLtfS**E1}N`uy*dQL!}7Yu-r0uZ8b|2JCWUbbyBuJ<fFn;Gp*LukGBcSDcBKZ~!sMMTnO$CDdw+4b@f}{MZJlb^9!$a8Mvfh@sXKLnY>y^9Y4gvneHt&_OxMiDTb=A}huId!-~D2V2W;%+3!rvU>+=WC2iU#$QJ%dUrO%l9=of`wU!GLgUaqX`Lfrt{VvUlP2}LaZ8qaCyb(5!J%?8=Tf7LV@g+gWMq-;-6i<htpuTgu&L3n=Ur2uLqX9)gP|C0daJ@oPpdNQ?T6EvseMOMWh(6FJPuVd86<>8Zv`DC9J^P1E~WD*A<W}T{o8WgGNw-l13T&tDQQbj9Ss;~v!nI%FVEF4Pn$P!|)&h)K84*J15@XbHyK0X=|5ims0Kf2zaVm(TDAlk34Zs1|VfN;Sw{a=%Yn`AB@2rR`5fY8)>*>WV%>;|ibfk(5K^<gLvMm3bnT}NTUl@w~&_Nv4etvq0{x)kM-7RS;rmo?wkOAR_`a=bM>TrC3_K=Rsq8T6|kFVW*wcdp@tL9FHj&Z!(^p3W%#gVIrT@xY3aSb9mijPCo+A?pGHcI_S@8@qP!2i2n2;B2V{e$v)-YKTy5q^6DTi^6mPxUuVnJG)u5ns%<fuWH*$ZmI_dT)*be=vI?sve_{k+T-1rjV2f;2wiQT<(>1)iWL$s2Gyijv5C`Eq4<`*fJ^}aRirpIc__=WKBf*y;{fTibNulQhFe>D(=NIC#q;lv4thTreUEM$#OkWk>@NAbsqCJ$R1ZlthNKMLX_Dx)m?XHOZBp$I55)>s5z)#9BQ8~J+X4JnV11)}uGpdhwkn73Zo`5i-g>nrSza+*Qrt#g6?t^heXLl%Thc*+rWRIX(}#aDuqOGRh^rCT=zU9Q4TaEK2DS<zY`Q}1L81CRvwbV_ZHI_mjfiUrhg)Zq)?S=WRFYQ^O{m|LC4(*TE2YQ7I9SxK;04~+d(beswYbW;3apfykaBzw&^!G1zIyj83P`${pelLD9yQ+<EY3sp>Q*;u6M^?}dYY0`WCdQwXeX$qowh9EVQcwd)FdsLJ*#n!$M>?*O~jV>sNPzpWK0vA(UlQu#UF})op-Wzoa($VobJfnW6mAGv{p-Oi}w!%5~7q%7E8`jVqZcAd;4217yAyT<I;)p*y#nB3qzxomgYqo8M>>jpt8^Q1IJYbp7p)%A9!6^%Pjnh?qu72A)0wbSv6p}TU`f|+09$icHf+4!mgwDqgxwz{_m|#SF}}Xw{`u~KQ}Gr)ou;fyUjbB^eWA6#D?K!@959JasE={isXKX`Ng7A+FPocaV7D&I3GW^PqD?_LE|(OXN5rMKa}T=ozIGLLQ$7gllJ*Fxj`1j{T(7nq>(J&x%IdHZ-z2lkpB@UT*d6?+u!`(ZvES>v5B}q_{jUuYWbOz+HPhBsi;OE#8&%l+<8HcOI9x$=<~JZ6~a0E+rz=@C#+`foI-P%+I?%-AFs_Nl&@=}^22sVPT|CA2pDzIqQx)nnmsjs{RU4BG@x0BKgRH@ZaZc(`_k@0N#=1#P0IOq&L==k@-2W-OyieMf1f1MUpxw;DnJ*x01<zKcQ;()c_OMQXs+tEt>LxsEMZZ#Py1D0#L(b*q|eb6u3`G(#$;K5X!3FyA?m72!>{?QwbG%DzKsU<FLZr+MXAdQAxk&8daHt5UzL>0pb3F>24{s_3zS%A%WZU(U4I*y(I-GHl;*%!^o3LsN^zx04d%#5(u-ghwXttyeXQxM*>Lv0EiJ@D=_MX1A@7uu8=~Yi9yM9rq+Ip~3!)`zeD5E>)0;N4UfhtZoibf{q;D)#JB$=7lq(TnrK7yf(cp0U_W??dWs*)y13WP%#B}(D33sWeAli7@a0MD)y#W%Qz%<T;1NG_jewJJ(vl#OZ(yoRkz0s*tWcalLs1=1J);f!*LU@Q3cK7Opr_A+yRwk}Wee?$N*e#MO(HdynsdV+&zV?qZ8RzjTN>VcB<NBaL>z2tX_SGJw)FerYX1s1qTS8h1#a5s(^@qC%s_vx9R}Xj&OD`eR(f}IQ7C%5Kt5lD4?mEdcj-}hEp2%;<nL8*wUiA>GL6-7cD4$9C9%&>p+)Vo_=NU=t4eg#+$RCpG)=Fl-D)mJAHj$SJdssQ^5GkMRh#Y!JF><%A1)(B`aXIa#s;WM#2igEasp1yQ2Gmj$WFd(y4+~>fBhgcHW!2xs_L`xoKEKk_($L)s0@Ms{rojBU&zDB2OZkS}M6il3vz$p3ahxuS>`3;AotntQ*4Am3VkUpwyRMoqwKc3JvpimADYrXBW3DD(m~~kl`4NnlrcYIFrf%)p=*ZpRU1?0Y>H;<mL!h_SK}g-X?-M9XIH4?kv+yR4uGk_TPV!`SA;TDOAu5^t4tqa5*LV4na$Gxz2Jr%~vBD9NuJa{av!#Lz-I#ZJ|LZ2f>+793xqxXUm(%!m4Wt^KQ7!T0JLgM5R&llTMvmo|j=X1pGR3Dek=9yX?cx+0x{^u^!kE=B{VENq9gTe%REi90MaaXuUpZ_V2!(yeQCVN;$u-YzEAVa^87>TpXnzD{n6TER+&?TS*_qx@(#P6p05sRIvg7DlW4^9{WPa5DbkzVyVbnI0$ybCf?kN5l15jgrV0l#&<Smq30M9A7g?i?URC>)pIcr;eOhkTUa#@1T6d1vbL4%8xGhJo{`KM(hMbJG4cs<7&gU=8g5CC!~G$A6&14{;8L>avSi>3>qTjLdG%RG)|h(0R`_9X})>1;`Ms=+z|x|%GD%6W61-4Mye;Z!DD%10`4Kc_tT*$s&*C~vqNh=xJ^iPu0K1%%-@BnAfXjah-={g|&`l-+vj7_A%YN0*u~@ljbuv~Xpfr1HkY&|WHqdYf60!$l1DYC!--q9VxrqTW%pUVlJ&)l9#SIaxkjx*Mt?pA?eH8Q#z>E#l(Mgx@ro#F<F4qTB(BR5f^81&Af&IVS4oKA<+zKljW|^5x{r*HtYNgYhyXl;UV|pFSOUm<^qyjcZi<&hABrFE7EyJ42oWg5vWmR_$&bG0ROA#HU&nZXwNAMcf%W%{>>F-2|dlcIzRT83*0ki$?;LJ$nc&Z`u$yVmQw^Lc@ekQ0<$iWFj1iTz>|%Kk`h-_kUP|<dcj7>re*50KqbwiWd~$@za}MK62J(l|&a9UOX-H3@^9Y{z18vaVycS=5(P$Sm{=X8jNdSp)RH7>M{4Nebc$RN|1_(+swsHhr&NmBuK<N0g5>$=Y*y}&sZ*9!>6Dj_o>}S;E!~x5W2|V?E@G@6EHYo_UY*6K)8G%=)pDm1A3L)&Bd<kb&u#3%3ZhUk;u5DE{yo`$IoH~<;g$GH=A^l(0Ru)x)97H`OOTlS<wZ7uU-MMM`DXS*6U6%U@A^8%7tuUE^Z&6m)#uxaksvcj#lDh#6F$kdqM^9(wDhN@5p7x!}J`}5Fkt7JK@P*qP%SKMTSRs2y37IjHM~1j5eG$THjJI%cCOrlwG0E8b~7nUSg48SeMMg>Wk}K@rv*Sg|vWgNV$%ki%pcvI{-+JfvHhi3|`yp*@q_3u`!f6$K<e8N=+A%n+n&49Cm2K%X$?_2wMB|$T=eB2IboUils+l0b>_ZEofvSBc-_Re3?zJLe|06uJFZcPV21_J@H4;U`j;<{zsIiM^}2H@RLiuvXU=rtz7Oc=`UC2$%?%nqVZ5xerf<dw}vK)Yw?k5nB7NDkVxK<F+8_zB6=E-EQBVVAp18aZp|izF)#AA!zqYPDX4SxFzEIp6{Q3a$T6j2_fbppQ=Q%)H64Ll!Xk>7#Bj=0=VV;^kv}<C#;h6Jw?TinNW?WAHI2(`)~0CiLd<)LU4i|Lw<jS0Jmf(yyGg-QC-yps&iGm+Sbkz(?1~9rwy*>Qii?tO14GT$=7q&}Fy?p8C#Bs)Ted#MB8xE^UQ&$g5rXhpyuw@PIYN-`?jlv|Vwp|!Js%)DC$NAqw2MjQoyeHPO)hTkg%u~@Vw$FS?^sT_hZ7ah;5D*P4idKZ+Zot29hjB2F{7*+GQ^eCp;`#aAS)XNQZ|#;Da&;*nb**+a!4f>08jwLOpx_J8cHYAGWc^EO*+2WuA->ARmzZozEQ?-OvqFAf0n6z2tQy?elXRY2G4|V=u~@ecDITSctgahb0+VYCJN>5X+4+hlfAk_P1C&#zaaPH9Q3cr;K3DtZ4?+3P~g%ct-nmiFRr0fw>2noOE;(9EP$7%pdFv|8Ug!Gjv(}v2Ovv-&zlM_Fw}P>5XEWf^GQ$7$_eHMI_Y&ki;0L)XZ119{Isa5vkNG#ZV<#sC=oVvfjQq@stLS!81&B@VbRYq)jSt(agq3im>v)`=5tGv;99+N^e&~blm5_uAv%Ivj{Ct`H>fzsx(o@FmGkjX&d#}FKFFD{riZ!Oht*T{vrURLK&JjzvaECB!n~qwKFu7(^vVAV4-DwCcd1MLN&^-{#HNIVLTUhyiQal+4v3SM7964S9@phy90<U80keE=BX4!J-Csw)`U(k0YRtaI!Vx|h%<dw-UWa^Vl6%sEfWq8YZOwPep70zgXEo8Ff$9B9x!|E@_g8*#tg-S}x_}ax+Uy+u{)!rN<&~!}_H{$GUcFkcQtdmaSS+&5a)ufH(>Tg?vY<IzOl?-<(>;$`W5IU?dPoa!$bGuNg)mqU_zH3vn4u-zQUG(yQabW7yo%1aCv6p_8;l>_aCn4(wxkPp#PwB^tutLFYq};hA>&U3Tq@mKNH~deJ|K!GzVly``sRHt?o-Fx2u6nL2wN?l2(L<bOQ~e=C2lhmC9wi;x<RWl0{~c?L5(98_Z^TqLbsEo)DC{3W`;Yrot|h^7v;nytn;XGQ$h=||IqdcGlX~23N`3;l3p*OB@z^v5W0vNPbQ%osBZ^=HE5LluSxf_z(8~&R+at&82K0VlS?o~YbK5Y2?5A6fCPvpFzq)Rs3Z#7f<=T=?0-~ueuYxM3GoXQtASdrYXg<9t+gO`0<{`ci<W}xzal;{=<S|ynrDG=NaP}Ws!!Y!s2>6pvn%q?X-@`j)+9;SMs_%+Fw?A9;GNU)(k)vE44Poc`C=%ebotI0@k3l^^Z<2{!RrzhC5q(e<p9B<ii8?K<(>Nhtum!&DnfW;2S|xYHY4j&v2ZV1dM>&K6i{CKgOB2>AIvRjU$4`0K1VV6I|G(E_aU!lDY-C7m)J13$s%^mF0r)*{fjZ(STa;q&Kv3ohFK!FJlKsx5eCW!cyrVWBi(7a=?0TDj%O51zl<Y_ug2^9NYbW61cu>)W(hZXHMEd>jyI`u^wCy|EVhU#%E{DS#$XBZ0|RD4C-sj8O1Cu#-4-v=sp#%avjY1_ZlnDabq(SLhG2m=m{zWNQOOi16n%aTnuFK^jd^8+8HI7VhTJA*8_6yylO3jbbP@N>Q`lpZL!g*`gH{*#Z|ey;#&|bxI9RtTyY3d+KBl5*;*{o2b#rxGY?h=}3K<`n1wDkqsb9^4&lftQlT{oX`jnT+ci6Xe)S++jti#$umKLMC&MnOFvNNQ0?cG~cliP(ZP5}dpotQoE5Gsp)v7$JSk|j#_cLL4|fPm_KFK~IK=1_dUwM|}+UaOXM`h9COT_l>J7|12R&oU{0ixV<B9vj%A=8CTDKu|tb(6ln1&lw<J-nD~TK0v=ODs;-;r4<7M=B;6xa6x=EjZ#DVE-f1z4(WnU{3^ycAcf}*PW-^5(SH=aVx4nXDSSl2VH{*(-TEmOR{b+AU31jOnAB4u-cy<>sPJfg^n?}uGeWJQu)-)d`9!^t7pNJps|;2nhV=p#*3ewnoUzm@A$5yT=me@T{SNEOE`75Ni1PM9dd)B@_v@*Y!A&1&@5ZwoA7KjQiHYckx`q#w?T|<YTkH0$v?ZaY&%!l^5C56xSa@!I7gH16gle8cvMlYq^{%1(gIC}2mW$xi_P1DCfLz0pD2<cWJ&@v;7E$DHDW|9vfq69Fs2d9X#)b5y0=vTw>g&ltrRdfeoVco-wBaXvL>+7tm*Dd}<jco)T4rko*-_=UXTW3JLw!`+yk$d{$+*G5Z2<7<#_hx1lEJ(~=E^%Re{UruVwf*Ua?*m+p)@d@(DiGoZNnfbZ8Yo;G^$ya#(3L<w26#RE$aiRKM|(zps()5Eob`HO4Lg%>wqu>pxQs`q0T=lWdimK+rXFjWpe3X27fMrFJD7T^l#O;C@^D3&?n-bartI=Vfduci&ganY^drhBn8I1@0^c-J_%-ILK`!kV<XSFL5&W5J<<M{M@u9Z#=(x^cFX(3;w9v3g7Kwv+8E~rvQl{fLBXc{!{8Vv6MH$yx1?cB`6y&t;V0LV>gr5(ERPCmo$OjCfPhf&?=Cy=3ceptDEU-~Vntn7nxJ09DiLcYrgiuh^TRuD=noR;YK%oS6##oYC}Dgx@#tt{0J0!D(@o%?qwWt!cAPe2i}cyMm{hNpBHWRmSB;+mlVpM8NuARxf+jPKRdg3SO!;Jj`oF4UJT@dND8Es`vz(PoL34tVrY&ktw5(Zr_3FpDDPhjC(w$RZYbunGH${ng1E}vPSsGvm?kTO5lg}*thzpqJa-s4OkZ*v<NYDTadQS&g3GkGcl%Z|ZBZD0o76@(fWU?V4Jt-Oh9OqBn{xczOe}J)LGB?VxxuSD*qRLT<4J|YBXxuu)le_h_4U(!T{K#=uQC@$>wh=wo*b>ElNaHq6X760N#8~P!qov0_qV55b5F#~v?aA;lY{dX`zkkQo?^~QMaB&JpiBm#{;jI%!d2Y+&{9bGXz62P_u`bSDNkqB;NkNoU4pb;Gk&(wxAet#hm2e0P%1E>);H}wVT;y9Rqq68C+-@;<6?~5Z1+XCy@Hq>1-Ti45)6qkt3Zod5;2-!$<m1r^@SzEX4W?=?s%T%GSrUnb!EaU+nue=YJM`v~h1Fyl$}_17maycSe%D1YqB*zUv2;1Ax3bbr7**7s9pTCuu5d8LBbsN!XANq@f9B=xtUQcxt$fQ?VCWAcJ`IpBjdB%rLKIZL&5}x18|g=#QBJBEf&uFf4cf+t_e)b5wM$0E4r?uFC?F#pohG;UzS!1AeM>bNq5SNUdGghW8675s#^2=Z@xzf(vq^awvEG0VKq$K0qUdsAq~Wviy{R;AjB|6@>blB^aC27)iEjeIxvXJj#XFYMix6Z~^oxcW%T{5@9_B8}truHm0Y23RVABjXRTmr(*<B}Qa8Wd1yW)rsg0;~aFT=w(hg9=a{KqY-2hY&;sYt~fqM(UpjB#N;*XSZD(RZIlNb^t~lBuM(4iM-Ir~0p&9Hl3h80!Kv7CbzkFlce7fgH{<d!Qnn6>;T2RtDDf!t<+BA`U|qc(u(k^ZqbT!;kW|_y8>=7^Ag}w!+<ogvZSa_tPKbQuy)9>(h6i-oE)O%$smu?cnWHouN*sIX>#X_~8&lHQ1{^kiqi#;0XT?4(QLp!HXXLJwic1z&N!4^bcW{U--@dJxwqRB>X*ip-X=#?c&dO7mpBAL&8(5UavcN-sfbWgZ>XMIMwJ6hu`zVQLlGE4}%{DRHolMcyZ7O6;Zqbxd*ksr!=Cy-uLq1=ukXh59whbKMf8I$nV3?{q@QoWB2F>3_R!_93J5x=qJ{E@dE+){(JuW{NQ;FOjGAXhSUB5w=RB(PCXZYj|kL#t36Jl+mGt96IB0l)NCyKmwMY^nJbprI<CKIDEW=nI(_^mdKto|aB0ojPT^=hq2ZSAZ1-yY)LN4*JN3z`_|#JKd`|GB>EH_65Nd~AMNAtO-?EcGXKk$1rM!oBeFEvJbdFWX4K=t$ahkym|4l|wVHWC;qnKRl*y<LOs@6^W#rEmbfwW0oTc^uOdhPy={21ID)Es5<<i9DKe#W9JR)j*bEFA?XvRovy4N1yKyZ;6Ss8++FfIPHORj_mV$VML$pcu2I(*au&H5S(pL>1nT<gN42{8$schP@$58q6AfcQF#$l1*ukSWx@tweLG1_9>vj``uFQ1FDCpVaNpeCJon;afQsY+_sIeh$9SJaXTknNAsIs-<^K?`9pAg^7-A*ADok)KYV`o;meoz&e@xfr^o+~RB6zFr|LYBc5lKIlMc+r7E{0@(Q%ESi?@OpTfg|b)W;~=rvqXz2097t;qL=+uFF5)yPx~d0#C-KbM^(6oI1wI%>*<)9}k;Wl$GPRWQ|=V23zanBl>6FMN6HWLoe?p!|%I?^Lu=IS3E6-ec&Jb)45v>U*I?QjsuuBXdjB7_JKixvOeq`%<jb)?pFI$^Z8sQB;qCv)voj3spFg`<ZAS@tER-mB^lb2fZ_yN95n|)$gcfPa0UG0b<B2vdQnN{COiQ3Th@ZNo?e%Z!clv65RLZ{^9cYko$kT>exH8A`+MJ6^Eb(H5U?~J#OZXgigI+jmXAWYMJecr`08PA&%EW+8X=GtRgo>r7?rAHFV!A0ggxF(9j%LSvRu4{(|JW=i=k+Bxb&zx*@vx@jnjtNPS$Zgx>KDKHLOv}vK^y2yrCp0vJHUb5s&WZ&+r+cAduV(pt=QLaj=YY3b(*Mza*<7#T@7pg7s#(^u$Olw;K|8{~&?;jr56|<)Aa>e~@5Zm#<{ImD3<}+=k5)k#ii!=<=bqH2MPrj`4sGJ_+Pj_hz7Z-IAg7r-nmQy2=z6q+YL#HX&e2C{h7nHV6A4mLrWrc48|k{WXsb%l@vx2sLb15rZPy-M6WEWpE@BVZcDxQoX4_1DqSYM!1O4GE>xKPhgPgEXrqGk1a9rtR>{CIncz^h9~xUU3)xM!wFu^Nd=8x%I;;Q=OJ5-#|A`x#=I3xHbfOMvWPhvDt4C<CYP)v_%eLEjALWpJVTaV5PZ-HP%y7T1xM{uXX8vuV`^)WVlz=<#b?pof(}t7_oCb;;*hmVp>fhmnxzTgVaUEpZ4Dszo^Gd>=LM#Q5Oc$~_7&AI2)-DQhD#R=R<7wAxd?JJvJ-dN9g<df096T;fU5saeUKF}0A#DeW(DYaCo9J|Ytch}4`AguD6(z6fSPs>t3k?oL1o|9Rfu{))$nS$ZqBmFBuR_leU|2#6V0MErwwqj>2#BeBaKN63llryqNDm%r8F?QfCz?<`^9-W(uWwm?jad<B8-N`dE#^%x_i~3)xjB4gC5Optpv#gcq4Il(5r&0tTZf6l-DvQRirLMD&ku!ojdq5d_DsiroPtqL%kfMyY;a90wtVI6F46A=q@5YFYon?mZQ7cm{=yVfixlKQLiav8^M`ZFGyQm^J^`-mDau{(oDN~=-e5bI-rd{><&QuC^|NI)R^g;IM_hx*#82hv!a>i!Tf{^pcF>-SxNSy<`31|C1gM?!!cYf&<y_EylgQ_mbR^ctBg}RIisvrbzd^tXTCEruEGfmG+?_3&vypaT?}bf@qFeO+sI@gBKq-}48WaKKTb4LiI}ON*|LSlB<b7?)BjE6q!pK{i<n)Mm&GGlthu9j@<k8o?5k&HCm@-+Q^6J*dVruhJ~Z|U&Ep`cbZBzvqx0M7?l)2fpu_^$I-P~!caK)2**W*&#@J8DQJyjPa@#xW3{bmp++vkof%ut37im^t@VxAR%+17yc~KfNKe351quV&32nTit<X7JQ{=ft5kW`1e`d(d7YixlIQmKO;9?i;TP_=lWrwHGV?&MdY6q#_5sRjCWS_hL7b9glX$Gu&--S-t8{UCfibT@CCMnQaDGTE^!!Ro}SS#_g};9u6QD^}O&3M)rYmJ(Dcz5#`MKsbs}><<0oA7Ah?Eayw`?jQg7add|@sCn`<a{d7|z6AY$pt@74Zkgr~8EWGKQ64H`|AgdwA_a}|zZ7ETo!4EyddvCT|NGtDv*7;TdGq1ZoA<9z-weqTF(Pu0PtYfNKE40@fpg2xAIkd&C7<<TTY&T)GTY<R?xn$M5DtazlGA6eXWlb8b8ui?ZS*nZap~($QV8}7Po!E41MGvYbO-V*B?se7_{m-8@f?Kpi}hwipwv&F-1z7vK42sP3>c*L>1Ms9*X9ea!c&wsbtj&>DQL9!FZJ(Lyvp)zIKxBSrR&{mzka(PG$*Lj%8E&$UdvbTTOe|S-Ddpm!;e3oJ11ZMdaTC3Nuz6M78B;ZbT$|a<eYr`;<$G;i1*&_=@_pLuI}MZ{|h^JhnBX~EiO6ox&nP+Fs3ZcXc+q^n~z&H79==&Fk(@t7abcsB{Rn>IwEp;1agAfIl;PF#PL!JeamQ*P8T>Ql*4b6`gEl%B0Ijgit{vH`c>$ZA{Nx|AV{+S=K(DuDf?_Lx1GvmRCSvdnvf#&H0?VEDSgJfuHh!{d#6en`QGlrqnu(?=A~nqy975GHOX_nFcB2|mA2<S?g0!@p%eK#-U{-!tk`A{6Q0g^VFl+UV*onk9YUvk!TiBUAgm}1r;w_ONG>~@^75;8v&)b!<f130;q<5p`L4SUJ=3L`lAMq)E>P|x&KOa0PEwRRCBExSk<SlVT!7;Jev|zTXMw4%aY-WIm7zP)vE37}d%!#@eL}oeL)`J1IgkC-wCCw&L3|gj2|lD4n)qU{D1a(nD)wYr)mVO%k}9$WOT*_vZL+XmsV++6{i&hYnH7c7yw6pM`a3SOC$CHcYD76Z9|8(O0;fiRF{cGay|P4U!U(K>>X`fIB6}7PHqzh-zlHZ&1x2`x+NKtwiz)T!;Qd<9NY*$q*BN@6EjOh|k``d@G7^J`NEj+y4OV@1zjKap8aZc#efu`%Qj0LiwhV5fd=+dcUxm82nz9IX9lS7<BXb2jl3Y0r8(v2{GXjgT2F46KtE4E*?0)-{^#EU&(G}?VLKLCf#DxfGZuiDZ#~T-bX^fcg{e}UW!{ZJbYhm}`m2*^Chs?Ume4ph~S`!=!C`i-A=*|LCUw~vTGrCm~iUL>hutjrQB(!`!swboIdLqx(Ic%<m0YgZ``Sv01b(L;CI5%c@UWv3jZIp>;3q@h$&K;h)AM<Pyr^%*ZhnmryfN4&Yo-ER*D&mlqhHzW~BN@_WV=y#RPH~m2*O(h7l>z3IWcX$MhGi~~te<>A>t@4`TB<?nY?{6$pToqelSd|_MY~R8;g%${{}MB(Fw5h0<f2G}Vcu#L1|7C42$V}4<$q=MR5FKW#etNy8xJZ$1&uO^SGzLBuo-HjX(d0VOFcqEtMl*9;OHAHOb=Q+G0keJ2vb8IWE0R%D`$g+)7IK)tX6(UMlM;pG}Gb;TvFe%+@2D-Mt42mP}<NqWd07E{(UWlmckcv-rG~l+JUj_482I$>g)*ngVlb&w`1+>`od~W1saxgrx_Hg++oFM{o#?cgR*4&JItSY<uG(e7(;>4uYqukEK8%_Ji~5(eqYorqSAM%=5DUBN0XLXA;t*K4~(*Ow)S=MEaM2Rp(Z1YuFA=U>$SggKFXdD)jJ7P#$(EcAQKxei4bB)Q8j~~FlNaVT<9pf&TeEb9l~`U8#?;*&aE5J=n4~0A33f$7wXQ0hBTQ*o$L{HRHr;MOCBffhE{zB<W-{pHBuw)k#TeygIF?<3F*}SRRMGs_BSMn)8AjAAi%nLQ*0dwaSq!B>7qeYo-ukOo^lY#bP;3hl{iBuJ{wS!NVbHHD=j-FIBAcf#A4`>4kUVmneb_|qqv9kvJeUWDM!h^4y@o?RcwH_aXw87G`SGJjB?KMTTAh{fL%UET_1x_l4(*d&5LcCrrVmuLxY4oZl0BlRLMEY^I7fRG~)zwol1&QGrm)k37Kzg<_w+!28{0L4<9#R@E0b9;C852qdVQX`<IkviI+w=K2MiXvZ}VMZNJ8colZEFF%5Y)!OjMPg-_H!M)RKs{4V1@n_^DI3{f9|ICw5bGQQI`DzLRxBMmRs6z}@$$aq0CHoZV2s|pbnv8vhJF`KgGFJTY24VK0&&!1?<^9|=O!u&_Gc$rM-8bi#Umu0Xhh=P3K1|I&yKEMbr&RYuztP^JDSJ`zyMjjdwtY?`(RWdN$ERtnxhZ#|n6>>7bi3NH^YW&uLvoLCNcLPPtNn~n{bvB9tv8!OmhjPRQ-pYFnE21wLZ9y?c)$h@*@jbe&eCMbQDq+NXi1gu=>yu53E3)%`r9gAkO8vemGn~0;zo|IFQLnxgB=>vNOfr}JfB#ete)tzmB^pp#XHo>DS0=M^tks<Kr35aLZ7vpa&w@~q@;zg;|IBTnbH0$R|A8<Jv5Z{~rQ<t)OgA6)v1^Guu`8RuLJ`qn0wfItm7TsShDmg>8KZz`kqbXH4n|$0o-iWpo3OOIweGn9Z@E>kMGGxHL?q=@{Cw&v>nD>AaZZ+YXH7?+Bn_SDJ#YctYx23IOC0ZN<D+lB%^J|>c6?{<kbD2m+z|KYzTs*qV6wO60KF~68|82*ek-5LgL$LD+8&Sc9&@5M0K!;!b38SC;E%zd?IA^u;jRj507VjLFgkjGTw+^{>6Xvy=#H~wXdnu;hNfR_GM5O{5M^e?Usuc|dsRgUqp|VQ4y|5zR>W^DT$w}NIu$^7?cJ#(N_(CU8slF5T5e_IarKcmnHWKA!I)(+;z%XoraMRx)1#$u`;X;mR})%X>qL%deZU+npZf~M>=4a+$E5_7@@qBEOLq{H%d2%f!)T{5t_`I4+z7K{_|S{I<gll{ZeYA6hkFTOM~d}A7D0s!Wg+SQw8f?b`9%{&*F4&dmLW(jNI5ORNokz8TeB<cE_X8-fl1b^qhDMe@zDzHv<m4hj(!?;{dyx}=X*^c#5V-f?e&{Vv<Il7<Rvr0drDz`Xy69TXc)Qpx?|{~#DilO<(da?e{3AR#ISD|e#M&pxo=SjVuvs=!%VC2#5%61@(;3-n8bb;X}5jIRqs{VhQMUaGa4eBO{=n@C@}i_?SmIPxk@9=!7F63Mqgt$#_;vpi}Bm*_`n1-uc(}X+ShNN0ydC`@`e-XpJ50<ti~~l23(wJ3uaL<YqRGDDs!h!g?*akXvB`V9}OxUc(y*reN>cCIxr|_5yHu4Hn^EsxdrO=gop<J51YqeL3msOyWbQ7gN9~hato6G7Ta}J*5yB6cEhuB%IcZaK^?+df^&1K*b0~pJg21K2kn@a#y3q{OabH5c)Po*%yK!p!+(d*P~2*n?#9Wkxbb}JoY%baTc@7Yf}p+mhPhWl*f-2Ms|o>Q>|^F!NPb5<XWNWQkVPLg>E9^wnM|%#(t5OvsL$2V5CsUcj~O3n%E-c4z<@xG!F<f*QUl^-v&i_e>Qxg-!HhhC5>&|><+nhb$T4CV7E3y0LEl|6M1v|1GLtDhn&(?OHk}iuZ>#qPjq#2wcz<qQ>}8yFoLBG3hO(6+>PD6OX8nWAZgdYBpH3>XIh`{OZHbYtZsO~fMt9uE_)q#as6U+x9qYr{`%iCL3ex>cf8z6qmkr>b$(xsSMe_owN*g;$n+-MC_lzl^js+(m-ma2d&w#oM(6nocRw~y?ox|RfU(CC!&e?6QQim?WI!Cr+*AednBn3xCo3f%)#`(%gR)BlgG2gi%PPp{xP;;;AoU|ggt3<E1H>c@DN$y&ssF)>FqL!YO2ZmgfMyB(j-*CoA5kVp>mrMiFDY~&RAG?!<!EZW}G@&UkVtwpoL2Zfz8^O!k(#lw{5&twn^yTQ8kXhh`E+q@FfMez9(Uf*X&&}V&pf!i6BnPU5dX0#TJy1x%RFu{zZpMX;j59$%LOKxiTMPs$0lyi&AhPLNv1X0&3taGzOFfcj^8}-q=p^v;-5+>u*_xXQd3U&o_Y~2HM_TBe%2_sjBSoj>*V}Lzk;=PiyIQNjvuw4>P!%RTBdaUB3mcBblo>l?r((;C*zzFOrhVu%9#xB&+fF@!jg?7D6(?=Ip|ss`n{$)H_fhxzp6!8iIy&^L7VQ`I8{2<$bl9ues(~hr4qw#l(iLZoQPsxZ;Okl|V@0Nojw)ZZk15?b)URF=Xt<h=+|Eaks5)nzYN48eee}FCNzz0Ky=!<M-Hs0I_fMZPveb>_<|?j}WmZ1CicgxBk<#ZpsjTFS`r$<XdG+AIUd)4D^J>=L*O&9?5z9H)xtxRE?)5zIofnT<Quyc7Bi40rRBg~$*(c3Qdsz9<D8Y|pqM(P13YH-A|Fl$~yknfvXGaMQtJ?G<F;F?IHK3u+hoOrJYFoIH$Tb8^Sfow~!F(L_zv3+)1z{JwrW`!2ZGAgEC-1dpTHfh&-ecPO&B_S;bczk$K_I+Ti{w_%w!<!?fiQ`&ZKBXORinI*?rWxZ1MXDcITaZjdE2qGexY3OeZy<sH|{&oa9o)xi!xq4#}rmQ%#B6=;vT`*19*K1iY$6Q99qJciy5}CDCTpb3|L3_>w8rlQ$-zNLagTMGBp<^&cYo@6<?_8BCpoLb5Z@MT77VEByw)ye}~jM`LA+)>=BIM5Ywf(FpTHb`kV^vK>WwU?hjP`zWN{ld|w4{L}d?N@PDG|zMemvyaVXk_X7dpg@SOda_SEok*N~gok0EiqWj{$K)J{w;u)32nC|I8l?d_2^ksx5r8k{TeRlE&#UH2_^>3%6(}uUeC2Ggl4V;1pQ!mE+g2q_?*u*q&Z}guXau7?QLs)Rj4vyq&7y>+`mJ>(AjR6fT=dht4Hrqj)5Mm__iJTmJUPC2=8HvTKmw7xlYLhg24doP9Vr0SW<ZXq3Ly1ED^Tq`bVk-Jr^m;!yPXSHq#hOLfdbw6Y-vqOfQL@gc5-?Tr6^)Z~&wm^wD|uBjJkD|S5KLmVbucn%(qpQ(_QlM#R0rhMm~-`xi%Nbvba;^ZU&yLlA#PGBjA(J&lF*u7P6S$u`h1holU+A2+cy=l%LtnGWS97CQJC*$!c)r1IoUDqN~G&U3vzpsoxpnmx1(jWn#>|c=23C;F0)IS=gRYJzYv|cFz=eyXENO(p%u+FM!B2rAuWJ}jr39A$S`k0l0zu_;|_j=?mr$<tzT7W+_gu;Y|3=TCK1Q5jI$ezN(orWx%Stt^Wn{}fea-!60QCC?sMl~pYhBbl!2cFc+`d=hFQiaWUN7R-ZQ9dq&~Bd;OWzWnFozBq5VL<4B}_rZA=;Tm<(ti;+uwyXQPIEXY4r9mh-Hd^31K|Jkz!R2lJji`If|IqET_e)0+9r&PUWR9k5z3eyb&TWqFyVFzFU?lD-oilEmfFJDIYKQHfJ@JnU(jk-_d6IjAH(Fig#Mf6&wFS#WQjH$AcPx5<MiJU=y;olc;BSMD}ihn`F8fZ}xhJj%k1YGO6yRj|)ynnPWJ$^;#SbuzgV2tem`{Y>zxVT;5qK9j`FPHdDhE?h*wgpuGDaVg^4<`kce&gQAZZEKq=-ck>UpHinlOv_=dp;H-BwozwO&Vv^n*b0R+x39}sM_U{kw1Z>n;<G#V%(xU$qI&`T=oHgM3>(RF{%z!(z4`Rz^mF*><n5dH$HR(8WrfAgJ9pTz`=PT*i_KcN1Y>UVq@d7!aabVXTX%qDrQHc~N(b>w1gwcPBHp9BP(;_zIC}rgl9>PxybZ0+<UT~+L`EO173oH^nQL_?PA`%)Mp=1q^t@x-haq(5N?{>U*kh!k^=l9bWzx6}NT-qyNh<5-2omebWD^wGJ;qG3_Sv2|{-c<<N~>B=s~q1veVXgMR^Q1xaAzBgU%(SeVqZiQ3hV0IA_1AKTyC9N1|nu0@U1%{MZu~oiD?<p5L&pO!8bRheZ@2G6RPeJc2a5``Z?U&ffLRBPB2<|y+PTNSZ86RwxG65jG*TFh|DzCP}lmrvw3R8;gf}%SY_+NToEu)$sBYSO#TS4Hz~*Q&J&VL5#>ZX*h#BJ*;y@f>$8O@FX9ky@y7e-u8k|XEGnjMCKXsg)sUk>C3@~2=NB7PT|d$ncNQ1ZJRw-4Fq~!6F!XV)R_KHS`YUF29I47@(am3Vg}3o?{i7`EnGNDz!_HDmCPxVZaY)j!^7v+WAsA?@e2U;pL$q(%1L2oqM*L$nfyu;2aadt0xZ+N#QC4*0^g79bM$#7i^yMJ@c=q#;?@r%fO>WVv9Bn-`2(^zJ8tn9}!&aE1Q3sJ<4PCR;p~Ke^^}5|7M3p}QCHqt78~eieZnl{snv6P6n|kMZwa#g+=|4aH{6XM!wZ~$rAM6@*2z7gyT9Wk1C@R|?^ajs+FM9pm)j5L+`m*q49GwEsQ^PXf;8LskoW{<F&FbTpa>tO*{>=jZ!Hg>Vly1gM&C=P?q&=5pKM2r%U?!(+=t;vQVp5lcTwUkO=;9?t&_twmid~<_^G!iSDnK448jDnPc2ryyLPg~~FEaIuo#*0#F#yQyCZND^Z06gBp)}=adX;83%XoG{hAtLi)Lz`zQlUVR;=w*b*HBXFXng3X^~?k@V<|~$=`f?fpa?1LH~_w&YYEbrTWkxwp!<5<AD)ZbJg;r;iEp@9n2In(Q$5LXNG`SThL>@$Pb)`5B_PL=DI&Tt_z+(rHcrmztRes%1lKJ=OO{cdUIKW!I4a@0w?IGxHsN|ejAjK2eQu{q(@~Lrp!mocG~TbK=_QXtvU%>2A(bGAzKr|=F>M`_H&QW8m=8_my}KYn-O&BGASG{R%k4{cz!o_+D1<{`qJRcnNjddEKy(4|8)qB+(J*B}*Gf#tNMLX-(Q(hXv^&M@i<=dw6WfQ)M!v`!J+y2`Dv(*-%mNBWNXn~Xl4g19_HILNQ|5#p-+eefb$&YjjCph3zdt@xnc7jx3~|CX9G^A5AY`RDbu9mhuBh!U<n3rEqF!`a+&!1cck>ilX|2swc0sV~SUD9j4{hh;@uyE6?g-c8eE{P+es|h2dPuK3zaF1`=rkH1#&-bm{@l5{-~?iwiUN*y0xfr=Ftb^?M&DKYc8~FvGoxhnqe^E{)%NAg4f)D6`O~|9sUz7XCfJ@P!t{gvJ;B&ihv-^9teJ)!=c)6C;*o)CY@KPG3%O04z?Ga1>~wVK$Xn24oZ8TNJHV|##n2iV%#sy{g+)s%u%jcAL&kX)tTLpHNJgcjQvgpux~5T{CvpDLnQkUA-=-bqcmv-uAxZT$#+!rKc??Q<5ZwDsEs&dK477O)3MO8_cQxvD>jiPhNNAO>o6I{iq_`XBOSpz*F3{47`@?h)Bp<ZkbMF2vT>t&R+kf6|oYLm&e>gg(ZeJjKWpn{kSg?9bRsi>~=b7%<Y9Pvyjj4ww=}O~}bVUnDoUxg%3lNG53x{-agn1)9&+-kHj^%dg#zO!@JtQ6+is-_T0jTN=Th*CPQDJ&2UaM-J@J+W=Y@NuE^O71(R_q0pc~aa%=|?tkLuj>$vsRVwVnjg(MH=Y)bh#;#>v*W|!Nmb83L$P{r+{&^EXkm83KvHpt7sO>WsR3;!Kn1Pr6X^QNC)MM2Fep@XnD5c2rTI7kUfB9V6)8g$Hry$spg;nzvDGE<y5YkywQmzD!#;=KET*1$`jK_iCS2Lm1BI;Hrugtfwy{5oIVFfslm$Zf`ySy0k_vn2eTjWhJya^p#hQ&K)*W8@0gf^fv`vjrl?>~qfJpyGQeg?GonI=b6E)pIbDLH4VAE5YnUGxH#p6c;aFH0gQqP`Bm{2~?HVD6g7Mcj82+%(y9JXzSO!m|?3+?W-Em4&>4~uqat&T>sym4d6EqZ`C)suUvJH!HQE16fSAf2%Fgnb^mO7j~M#2r~q}SL+$R7byE#zq!c*_uvD;8#)S{|s#b4j89d_SRH`PwdS<Rc%@B8UFSu+w50_LorxY9s}Lc1(U7t&(M#rAdVO2-eK1pp@M4&c$*QPtdSOX${&MGb}5MIFC(kQmm7-OS=ZOh0lFqb8%M5ZLE3Oe~vhx^M8&Gnm8!Nb(2fKNHJ5qbdCr6p9d)0mVl1Dt+1&8&?8%-c~aPRXct720%|pOAH8fSuNkG1DCDk$sh&d*XMvoFAcvlA3Ug#UkTQ=r=wRFJG}G{*X%bkGz=|f1?q!ywF3WUU`m$<z)kD`~5Xo}HP-T)u5zMZ0uaN{3fij+eutsfLsf6+R-p84-l!#9KAqi1tl-Fs1>D1QnSLQ?1{DN(`mfcKE6u#+7^9E*_YPB@oD+)q&NWBi3&mv2Ce1_Rm%H?J_cdeCca*mm@3$#?Q{g4&puJdD(7sALbI2Rifi2U!9k*pfDa+Bp(AUNCz27a9YNke*Lu$`9(3~4AbZ<4TTRs%DwO>_F#dFK-vJYcI<GdkN56e7BX68^Y%Z$A9=?!%k#mp5mh-u?XH-i~;lcf^d0?#$)67uk^cB|a)m1ands+ZB50Z^IJP)@B#mP>Ahl9Cp>o5b=vq%cWjs+(WVBoJS-^dX#$kq!HE7ZlH0?*rDv{H`?uYNT8&nqneJ2w<Eo8@a{IdqnH-<2g-GZRoB`jbAS<d%(x8RX5IvJb}OW53wHeicHBfgOc#kY*j_@=y~az}tIi(U?CPvGfK8KF8BTEoUh4?P)W{ebJwG>nN|nA)O>NGn6(awf1axlAqUEN7a(QJ4k|B>C>n!8#oMJ<R{W&!($XfbPR@|=hDY^!lR?4l7W~P_YgzHpXW+?x5r}26YU{o~=up{lwbn4P06Ba%8_2YmayIn)zYu*NIR36qQR$wr;Q}qFALyc-bb;O=MUe$kX>+X0>NYQj|Z1&6Gt-o`rE6dC}X(filoHViH|H<u7SZzp&m8spaDb8(*hEp=Mg35lkWo)m0sC`$-a>mEYL;LL7_Fa?bxgb4L$lK7dxlMCvar!%Stlw_MFRqg_Ux#vcHB!}CZ_5QWh}LM)e3$PfL3i`IuU;qJtkzq;;5L*U4#S3m#WbOU7qGF)GIzr;Sy6_`&=ub>lZOEVg_li=#{0BW8N_{MOqJvKVO>q~YdP}i>~3gn=4zn|cg4~i2*<yxUa0y|+$cy`h}VwRXf0kW#g@xT&L<020l>x*npZOgRsB-YMJ2R+&~#_S_V(KJ%htZIH-hemU@Lk{7>IFS-2&r|3>X%qKqBfn=T}0qB2d+8^{#D^NbEo%!kU?}c-uLz&(VEDnj1s33L8973cEA(eg>#sxcC?r(L9ze7=X6uXc=C`+k$g@IV@VIQ7Z0;X=oqw!WKl_Fne@EZ>oM4KE=jW4p|!Njdh*#?mAn$9W!#V!`JzGO+FY8b^>dU&#UJ=)<6JMV@HK=$Rj$WhB1rLD4V1Siu$rc?a0&6dkwT)t%>uo{{WMG=NV{_@S)S-!X(QBvfVt=jY*+MBZka<Iw4QX(O?I9WSFf8NDEA)?t$vCS3BAG@ATI^17X`ZSp>4KDk$xu@pMX_LHSNYge)>+K_f<zZm`-5>m7!u35OvGKVgVUFp%q^bgc!I7|Q-1BwIH$', 'NB_FLUX_REQUIREMENTS_LOCK.txt': 'c-l?P!485j41nSLo}vl&g+YkMWGCano3Aiu36rfOT_N)J4o@EYr~TT%?Sp<5d;#N#jN}oJw!oPZl3d{W*k#HDy*8aTy=vf2MO=&^6BdLGxr9sw0)Yb`(a}EulB>LMeyvrmC$I#f`mUrr*TC+LUNY}m(VCXs+D`X+P{oI1Fxs58QS=-$Cnwep+L&MPF;w2#_6V)2+cPp@wDweiR(%0c|466'}
EXPECTED_SHA256 = {'NB_FLUX_SPIN_RESOLVED_engine.py': '676A2C19A5A817178903B6A7AC27A5D3FDC8C0B3080DC92EDB482456A43079D2', 'NB_FLUX_T1PM_engine_repaired.py': '258FEFD36B4155D1E96376A7FFCC4282B810BD2F1279B90DD58A8727FABCB42C', 'NB_FLUX_REQUIREMENTS_LOCK.txt': '92A500D07039716E47A0B5BD96E1AFDA74DE76D8CEA274CB46F9F458B66FD282'}
PORTABLE_PACKAGE_MANIFEST = {'schema': 'nb-flux-colab-tuning-chain-v1', 'payload_sha256': {'NB_FLUX_SPIN_RESOLVED_engine.py': '676A2C19A5A817178903B6A7AC27A5D3FDC8C0B3080DC92EDB482456A43079D2', 'NB_FLUX_T1PM_engine_repaired.py': '258FEFD36B4155D1E96376A7FFCC4282B810BD2F1279B90DD58A8727FABCB42C', 'NB_FLUX_REQUIREMENTS_LOCK.txt': '92A500D07039716E47A0B5BD96E1AFDA74DE76D8CEA274CB46F9F458B66FD282'}, 'action': 'bounded real spin-resolved tuning chain', 'physics_interpretation_authorized': False}
PORTABLE_PACKAGE_SHA256 = '69A3F3A45BEA5509121D78AD0F9E8A2A7850A6A51B1C939F12121473374FAC14'

runtime_dir = Path("/content/nbflux_embedded_runtime")
runtime_dir.mkdir(parents=True, exist_ok=True)
for filename, encoded in EMBEDDED_PAYLOADS.items():
    decoded = zlib.decompress(base64.b85decode(encoded.encode("ascii")))
    actual = hashlib.sha256(decoded).hexdigest().upper()
    if actual != EXPECTED_SHA256[filename]:
        raise RuntimeError(f"Embedded {filename} failed SHA-256 verification: {actual}")
    target = runtime_dir / filename
    temporary = target.with_suffix(target.suffix + ".tmp")
    temporary.write_bytes(decoded)
    os.replace(temporary, target)

sys.path.insert(0, str(runtime_dir.resolve()))
import NB_FLUX_SPIN_RESOLVED_engine as engine
assert engine.module_hash() == EXPECTED_SHA256["NB_FLUX_SPIN_RESOLVED_engine.py"]
assert engine.base_module_hash() == EXPECTED_SHA256["NB_FLUX_T1PM_engine_repaired.py"]
assert engine.requirements_lock_hash() == EXPECTED_SHA256["NB_FLUX_REQUIREMENTS_LOCK.txt"]

def ensure_a100():
    try:
        import cupy as cp
        count = int(cp.cuda.runtime.getDeviceCount())
    except Exception as first_error:
        print("CuPy is unavailable; installing the CUDA 12 wheel for this Colab session...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "cupy-cuda12x"])
        importlib.invalidate_caches()
        try:
            import cupy as cp
            count = int(cp.cuda.runtime.getDeviceCount())
        except Exception as second_error:
            raise RuntimeError(
                "CuPy cannot see CUDA. Select Runtime > Change runtime type > A100 GPU, restart, and Run all. "
                f"Details: {type(second_error).__name__}: {second_error}"
            ) from second_error
    if count < 1:
        raise RuntimeError("No CUDA device detected; select an A100 runtime and Run all again.")
    device_id = int(cp.cuda.runtime.getDevice())
    props = cp.cuda.runtime.getDeviceProperties(device_id)
    raw_name = props.get("name", b"unknown")
    device_name = raw_name.decode(errors="replace") if isinstance(raw_name, bytes) else str(raw_name)
    if "A100" not in device_name.upper():
        raise RuntimeError(f"This bounded run is authorized for an A100; Colab supplied {device_name!r}.")
    return cp, device_name

cp, device_name = ensure_a100()
probe = engine.base.Backend(True, 20260822)
runtime = engine.runtime_manifest(probe)
print("Device:", device_name)
print("Engine SHA-256:", engine.module_hash())
print("Base SHA-256:", engine.base_module_hash())
print("Portable package:", PORTABLE_PACKAGE_SHA256)
print("Discovery runtime:", runtime["python"], runtime["numpy"], runtime["scipy"], runtime.get("cupy"), runtime.get("cuda_runtime"))


## Locked bounded configuration and persistent storage


In [ ]:
# This is one real but deliberately bounded tuning chain, not a production ensemble.
cfg = engine.SpinResolvedConfig(
    beta=6.0625,
    L=16,
    Nt=20,
    thermal_cycles=200,
    n_cfg=48,
    separation_cycles=4,
    overrelax_per_cycle=2,
    proposal_size=0.30,
    target_acceptance=0.56,
    monitor_every=10,
    ape_levels=(0,),
    loop_shapes=("P",),
    bootstrap_samples=100,
    fit_tmin=1,
    fit_tmax=4,
    seed=20260822,
    prefer_gpu=True,
    install_cupy=False,
    cold_start=True,
    published_asqrt_sigma=0.19472,
    published_asqrt_sigma_error=0.00054,
    reference_asqrt_sigma=0.19472,
    reference_asqrt_sigma_error=0.00054,
    scale_source="Athenodorou-Teper-beta-6.0625-table",
    physical_radii_sqrt_sigma=(0.30, 0.45, 0.60),
    campaign_id="a100-colab-spin-tuning-v1",
    chain_id=0,
    chain_count=1,
    checkpoint_every=20,
    topology_every=4,
    spatial_flow_step=0.025,
    flow_refinement_tolerance=0.02,
    topology_radius_sqrt_sigma=0.50,
    topology_flow_step=0.025,
    contaminant_radius_sqrt_sigma=0.45,
    scattering_shells=(0, 1),
    min_physics_blocks=200,
    min_blocks_per_chain=25,
    radius_relative_tolerance=0.12,
    enforce_runtime_lock=False,
)
cfg.validate()

# Drive makes the two-slot checkpoints and immutable raw chunks survive a Colab disconnect.
try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    output_dir = Path("/content/drive/MyDrive/NB_FLUX_A100_SPIN_TUNING_V1")
    storage_mode = "google-drive"
except Exception as exc:
    print("Google Drive mount unavailable; using ephemeral /content:", type(exc).__name__, exc)
    output_dir = Path("/content/NB_FLUX_A100_SPIN_TUNING_V1")
    storage_mode = "ephemeral-content"
output_dir.mkdir(parents=True, exist_ok=True)

tag = engine.chain_tag(cfg)
pointer_path = output_dir / f"{tag}.checkpoint.json"
resume = pointer_path.exists()
print("Storage:", storage_mode, output_dir)
print("Chain tag:", tag)
print("Resume existing verified checkpoint:", resume)
print("Bound: 200 thermal cycles, 48 saved configurations, 4 separation cycles, topology every 4.")
print("Radii rho =", cfg.physical_radii_sqrt_sigma)


## Run or resume the real chain and verify every committed raw chunk


In [ ]:
summary = engine.run_spin_chain(cfg, output_dir, resume=resume)
if not summary["complete"]:
    raise RuntimeError("Bounded chain stopped before all requested configurations were committed.")

verified = engine.verify_raw_chunk_records(
    output_dir,
    summary["chunk_records"],
    expected_tag=summary["tag"],
    expected_config_sha256=summary["config_sha256"],
)
if int(verified["stop"]) != cfg.n_cfg:
    raise RuntimeError("Committed raw ledger length does not match n_cfg.")

streams = engine.read_raw_chunks(output_dir, summary["chunk_records"])
finite = all(np.isfinite(value).all() for value in streams.values())
topology_mask = np.asarray(streams["topology_measured"], dtype=bool)
topology_values = np.asarray(streams["topology"])[topology_mask]
acceptance = np.asarray(streams["acceptance"], dtype=float)
plaquette = np.asarray(streams["plaquette"], dtype=float)

qc = {
    "schema": "nb-flux-bounded-tuning-qc-v1",
    "scope": "one-chain tuning data only; no mass, residue, or continuum inference",
    "passed": bool(
        summary["complete"]
        and finite
        and int(verified["stop"]) == cfg.n_cfg
        and topology_values.size == 12
        and np.all((acceptance >= 0.0) & (acceptance <= 1.0))
        and "A100" in str(summary["runtime"].get("backend", "")).upper()
    ),
    "portable_manifest_sha256": PORTABLE_PACKAGE_SHA256,
    "summary_sha256": engine.sha256_file(output_dir / f"{tag}.summary.json"),
    "tag": tag,
    "storage_mode": storage_mode,
    "complete": bool(summary["complete"]),
    "configs": int(summary["progress"]["configs_done"]),
    "thermal_cycles": int(summary["progress"]["thermal_done"]),
    "raw_ledger": verified,
    "all_streams_finite": bool(finite),
    "channel_shapes": {key: list(value.shape) for key, value in streams.items() if key.startswith("channel_")},
    "operator_labels": summary["operator_labels"],
    "plaquette_mean": float(np.mean(plaquette)),
    "plaquette_std": float(np.std(plaquette, ddof=1)),
    "acceptance_mean": float(np.mean(acceptance)),
    "acceptance_min": float(np.min(acceptance)),
    "acceptance_max": float(np.max(acceptance)),
    "topology_measurements": int(topology_values.size),
    "topology_mean": float(np.mean(topology_values)),
    "topology_std": float(np.std(topology_values, ddof=1)),
    "runtime": summary["runtime"],
}
qc_path = output_dir / f"{tag}.tuning_qc.json"
engine.strict_json_dump(qc_path, qc)
print(json.dumps(qc, indent=2))
if not qc["passed"]:
    raise RuntimeError("Raw tuning-chain QC failed; preserve files and do not interpret the data.")


## Package and download raw results plus the active checkpoint


In [ ]:
summary_path = output_dir / f"{tag}.summary.json"
pointer = json.loads(pointer_path.read_text(encoding="utf-8"))
active_checkpoint = output_dir / pointer["checkpoint"]
raw_paths = [output_dir / row["path"] for row in summary["chunk_records"]]
delivery_files = [summary_path, qc_path, pointer_path, active_checkpoint, *raw_paths]

delivery_manifest = {
    "schema": "nb-flux-bounded-tuning-delivery-v1",
    "portable_manifest_sha256": PORTABLE_PACKAGE_SHA256,
    "files": {path.name: engine.sha256_file(path) for path in delivery_files},
    "embedded_payload_sha256": EXPECTED_SHA256,
    "resume_note": "Extract beside the portable notebook payload; pointer selects the included active checkpoint.",
}
manifest_path = output_dir / f"{tag}.delivery_manifest.json"
engine.strict_json_dump(manifest_path, delivery_manifest)

zip_path = Path("/content/NB_FLUX_A100_spin_tuning_results.zip")
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=6) as archive:
    for path in [*delivery_files, manifest_path]:
        archive.write(path, arcname=path.name)
    for name in EMBEDDED_PAYLOADS:
        archive.write(runtime_dir / name, arcname=f"provenance/{name}")

print("Result bundle:", zip_path, f"({zip_path.stat().st_size / 2**20:.1f} MiB)")
print("Persistent checkpoint/raw directory:", output_dir)
try:
    from google.colab import files
    files.download(str(zip_path))
except ImportError:
    print("Not running in Colab; automatic download skipped.")
